In [ ]:
#!/usr/bin/env python3
r"""CAD Cardiac MRI — University Research Portfolio Pipeline, Revision 2
===================================================================

PURPOSE AND SCOPE
----------------
This is a single-file, institution-neutral research implementation for patient-
level classification of the Normal/Sick labels in a public cardiac MRI JPEG
release. It includes the research question, data contracts, mathematics, runtime
instructions, implementation, output definitions, and interpretation limits.
A high AUROC is an association with this released cohort, NOT proof of CAD-
specific image interpretation, prospective clinical validity, or causality.

The research question is:
    Does cardiac-region information outperform exactly matched controls, and can
    a fold-local mathematical nuisance penalty preserve discrimination while
    reducing measured score dependence on acquisition/export variables?

The public release was previously confirmed by its author to contain 30 patient
units: 16 Normal and 14 Sick, with 63,425 images in the user's earlier runs.
Those images do not create 63,425 independent labeled observations. All series
and slices of one Directory_* remain in the same supervised fold.

WHY THIS REVISION EXISTS
-----------------------
The user supplied a previous compact-pipeline run with:
    BASELINE_A17: main AUROC 0.9732, repeated median 0.9621;
    guarded custom logistic: main AUROC 0.9509, repeated median 0.9643;
    full nuisance projection: repeated median 0.7567;
    outside-support control: repeated median 0.8281.
These are HISTORICAL observations, not results generated by this revision.

The earlier audit found several issues that this revision addresses explicitly:
* The compact pipeline counted 2,879 series proxies, versus 2,859 in earlier
  full-suite logs. Immediate-parent grouping can split nested folders differently.
  Both grouping policies are now enumerated and saved, not silently conflated.
* The console combined repeated median AUROC with a single main-run nuisance R2.
  Every repetition now computes and saves its own nuisance diagnostics.
* The old nuisance probe trained on global outer-OOF score targets. Generators of
  those targets could have trained on the current probe's held-out patients.
  New probes use ONLY score targets generated within the current outer training.
* A guarded result could select lambda=0 and therefore contain no nuisance penalty.
  A separate no-guard ablation now fixes lambda=0; guard candidates are positive.
* Full residualization could remove genuine biological information along with
  nuisance. rho now selects partial removal, and export metadata are separated
  from potentially anatomical control representations.
* The purported Bayesian prior canceled algebraically in the fusion formula.
  Fusion is now honestly named reliability-weighted logit pooling, with no prior.
* Larger tau increased relative long-series weight, contrary to the old comment.
  The corrected tie-break prefers SMALLER tau and includes tau=0 (equal pooling).

Reusing the same explored patients after changing a protocol is an internal
sensitivity experiment, NOT independent prospective validation. Changes to series
membership and training/evaluation rules also mean that differences from the
old run cannot be attributed to one mathematical modification in isolation.

EIGHT EXPERIMENTS, FOUR FROZEN IMAGE VIEWS
----------------------------------------
1. BASELINE_A17
   Compact A17-like hard-support cardiac intensity reference. Its preprocessing
   and capped PCA are preserved from the compact script, not claimed to be a
   bit-identical reproduction of the much larger historical full-suite A17.
2. CONTROL_SUPPORT_ONLY
   Exact binary support used by the reference, without original MRI intensities.
3. CONTROL_SHUFFLED_INTENSITY
   Same support and same normalized intensity multiset, with deterministic affine
   permutation of visible pixel ranks. This is a spatial perturbation, not a
   uniformly sampled random permutation or proof that every spatial cue is gone.
4. CONTROL_OUTSIDE_SUPPORT
   Exact retained-content complement of the support, independently normalized.
5. CUSTOM_NUISANCE_PROJECTION
   Ridge residualization with selected partial-removal fraction rho in [0,1].
6. CUSTOM_LOGISTIC_NO_GUARD
   The SAME custom optimizer and L2 grid as experiment 7, with lambda fixed to 0.
7. CUSTOM_MINIMAX_CONFOUNDER_GUARD
   Custom logistic objective plus a strictly positive linear covariance penalty.
8. CUSTOM_RELIABILITY_WEIGHTED_LOGIT_FUSION
   Patient training and inference both use reliability-weighted series pooling.

Only four image representations are encoded. All extra mathematical experiments
reuse the same frozen series bank. No large architecture sweep is performed.
The ordinary reference still uses scikit-learn's liblinear logistic regression;
it is NOT a solver-matched ablation for the custom optimizer. Experiment 6 is.
Both custom variants share the same hyperparameter selection procedure; L2 may
still be selected differently, and all selected values are saved.

DATASET CONTRACT AND SERIES GROUPING
-----------------------------------
Input root:
    Normal/Directory_*/.../image.jpg
    Sick/Directory_*/.../image.jpg

patient_id = Directory_*; labels are 0 for Normal and 1 for Sick. A Directory_*
identifier cannot appear in both classes. Empty patients and unreadable images
raise explicit errors instead of silently changing the cohort.

Default MAIN_SERIES_GROUPING = "patient_child":
    Directory_1/SR_1/subfolder/image.jpg -> Directory_1/SR_1
Every descendant of one immediate child of a patient is collected into one
folder-defined proxy. Images directly under a patient form __ROOT__. This is the
rule in the available historical full-suite source.

Sensitivity option MAIN_SERIES_GROUPING = "immediate_parent":
    Directory_1/SR_1/subfolder/image.jpg -> Directory_1/SR_1/subfolder
This reproduces the previous compact grouping rule. It is not selected by AUC.

The pipeline saves both assignments for every image and both per-patient counts.
It does NOT force the total to 2,859 or 2,879. Exact attribution of the historical
20-series difference requires the actual manifests, not just previous totals.
Neither series rule recovers a validated DICOM SeriesInstanceUID, view, sequence,
cardiac phase, or acquisition protocol from a folder name.

FROZEN IMAGE PIPELINE AND SHAPE TRACE
-----------------------------------
    JPEG -> grayscale pixels
         -> conservative label-blind dark-border removal
         -> fixed-content 240-in-256 canvas, preserving aspect ratio
         -> raw uint8/255 view, separate min-max MONAI view, content mask
         -> frozen pinned MONAI segmenter
         -> hard foreground mask and plausibility gate
         -> fixed dilations OR fixed central fallback
         -> intersection with content mask = one exact final support
         -> four matched views -> ImageNet normalization -> frozen EfficientNet
         -> 1,280-dimensional embedding per slice
         -> streaming exact mean per series -> equal mean per patient
         -> fold-local learning and nested patient cross-validation

Relevant tensors:
    raw/min-max/content views       [B, 1, 256, 256]
    MONAI logits                    [B, 4, 256, 256]
    hard support                    [B, 1, 256, 256]
    four RGB-copy views             each [B, 3, 256, 256]
    aligned EfficientNet inputs     each [B, 3, 224, 224]
    slice embeddings                [B, 1280]
    each series feature matrix      [n_series, 1280]
    each patient feature matrix     [n_patients, 1280]

MONAI ventricular_short_axis_3label is pinned by repository revision and SHA-256.
Its classes are background, LV blood pool, LV myocardium, and RV blood pool.
It is a short-axis ventricular segmenter, not a validated whole-heart segmenter
for every sequence/view in this heterogeneous dataset. The plausibility gate
is an engineering safeguard, not clinically validated segmentation quality.
The exact support complement can contain atria, vessels, or missed heart tissue.

EfficientNet-B0 uses an explicit IMAGENET1K_V1 checkpoint with the ImageNet head
removed. MONAI and EfficientNet are frozen/evaluation-only. Neither sees outcome
labels. This is operationally end-to-end from files to scores, not joint neural
end-to-end training. No claim is made that ImageNet textures are cardiac-specific.

REGIONAL NORMALIZATION AND CONTROL CONTRACT
-------------------------------------------
The visible region defines the robust percentiles, not the full image. For a
fixed support, changing hidden outside pixels cannot affect normalized cardiac
pixels; changing hidden inside pixels cannot affect the complement's scaling.
This statement CONDITIONS ON A FIXED SUPPORT. Recomputing segmentation after
changing an image could change support location or shape and thus the output.

A fixed masked histogram approximates the 1st/99th percentiles. A minimum range
avoids extreme amplification of low-contrast regions. Tiny/constant visible
regions can produce zero images. Their frequency is not proof of pathology.
Support-only retains geometry, and the spatial shuffle retains intensity values
and mask geometry. These controls can themselves contain anatomy and protocol
information; they are not all purely non-anatomical negatives.

SERIES CACHE AND MEAN POOLING
----------------------------
For series s with n_s slices and embeddings e_i:
    mu_s = sum_i e_i / n_s.
For patient p with S_p series, by default:
    x_p = sum_s mu_s / S_p.
Streaming float64 sums followed by float32 means reproduce this aggregation up
to floating-point rounding without retaining every slice embedding in RAM.
A long series does not receive more patient-level mass merely because it has
more exported frames. Each series proxy still contributes equally, so changing
proxy boundaries changes the patient representation.

A new cache schema includes the grouping policy AND every image's patient/series
assignment, as well as feature settings and file size/mtime. It is a practical
cache identity, not a byte-for-byte digest of all source images before decoding.
Confirmed decoded-pixel duplicates are audited during extraction. Cache hits
also validate the current manifest and enforce the stored duplicate policy.
Changing only classifier grids/repeats does not force repeated neural inference.
Changing series membership does: old compact caches are not silently relabeled.

A unique run folder holds reports. A sibling feature_cache directory is reused.
This prevents old seven-experiment outputs from masquerading as current results.

OPTIONAL MANUALLY REVIEWED SEQUENCE/VIEW ANALYSIS
----------------------------------------------
The pipeline always writes a CSV template containing series IDs and empty review
fields. Supply a completed CSV through MAIN_SERIES_ANNOTATION_PATH or the
CAD_SERIES_ANNOTATION_CSV environment variable. Required fields are:
    series_id, sequence_type, view_type, contains_heart,
    is_localizer, is_derived_export.

Only explicitly cardiac, non-localizer, non-derived reviewed rows are eligible.
Unknown/duplicate series IDs fail instead of silently mapping old-grouping rows.
Blank review fields remain unreviewed. To avoid label-driven matching, retain
only sequence/view cells present in EVERY patient. If no common reviewed cell
exists, the optional controlled run stops with an explicit report; no patient is
automatically dropped to make the result look stronger. Ordinary runs without
annotations record SKIPPED_NO_ANNOTATIONS and continue.

Annotated pooling is:
    slice -> series -> equal mean within sequence/view cell -> equal cell mean.
The reliability method uses the same equal-cell policy, with its weights applied
only within a cell. The template omits outcome labels, but patient IDs/paths may
still reveal provenance; it is not claimed to be a fully blinded clinical review.
Nominal cell matching does not certify identical scanner or acquisition settings.

SEPARATE NUISANCE BLOCKS
-----------------------
export:
    native height, width, aspect ratio, bytes per pixel, crop/content fractions,
    number of series and slices, mean/max series length.
anatomy_proxy:
    fixed random projections of support-only and outside-support embeddings,
    MONAI-valid fraction, support-area fraction.
combined:
    both groups, retained for comparison with the old diagnostic concept.

The random projections use a fixed seed and learn nothing from labels or cohort
values. All subsequent learned scaling/PCA/regression remains fold-local.
The default training penalty uses ONLY export metadata. The anatomy_proxy group
is audited separately because it can contain genuine biology. No block is a
certified causal confounder set: acquisition and biology can be correlated.
For an explicitly labeled sensitivity run, regularization_nuisance_scope may be
changed to anatomy_proxy or combined. It is not tuned using outer-test metrics.

CUSTOM MATHEMATICS 1 — PARTIAL FOLD-LOCAL RIDGE RESIDUALIZATION
------------------------------------------------------------
For centered cardiac X_c and training-encoded nuisance Z_c, estimate:
    B = argmin_B ||X_c - Z_c B||_F^2 + alpha ||B||_F^2
      = solve(Z_c^T Z_c + alpha I, Z_c^T X_c).
Then transform:
    X_clean = X_c - rho Z_c B,        0 <= rho <= 1.

rho=0 removes nothing; rho=1 is full residualization. alpha shrinks the nuisance
regression coefficients and is NOT itself the fraction of information removed.
Means, scalers, PCA axes and B are fitted on training patients only. Validation
uses the frozen subtraction. At rho=0, only centering remains; the downstream
training scaler absorbs that constant shift, apart from numerical roundoff.

rho and alpha are selected inside inner CV. AUC is protected by a predeclared
near-best frontier, then nuisance score predictability can break near-ties.
Removing information predictable from Z can remove true disease-related signal.
Neither ridge algebra nor a low surrogate R2 proves causal deconfounding.

CUSTOM MATHEMATICS 2 — LINEAR NUISANCE GUARD WITH MATCHED ABLATION
---------------------------------------------------------------
Let s = Xw+b, and Z_c be centered fold-local nuisance coordinates. The loss is:
    balanced logistic loss(y,s)
    + lambda || Z_c^T (s-mean(s)) / n ||_2^2
    + (l2/2) ||w||_2^2.

For covariance vector c = Z_c^T(s-mean(s))/n:
    max_{||a||_2 <= 1} Cov(s, Z_c a)^2 = ||c||_2^2.
Thus the penalty controls the strongest linear nuisance direction under the
chosen coordinate norm. Scaling of Z matters; nonlinear dependence is not
constrained. PCA coordinates are not automatically a causal invariant space.

The code implements balanced weights, analytical gradients, batch gradient
descent, and Armijo backtracking. Centering Z explicitly makes the gradient valid
for noncentered numerical inputs too. Accepted losses and iteration counts are
recorded; reaching the iteration cap is not claimed to be a proven optimum.

CUSTOM_LOGISTIC_NO_GUARD uses lambda=0, the SAME optimizer/L2 grid/preprocessing
and inner selection rule as the guarded experiment. The guarded grid contains
only positive values. L2 can still be selected differently in the two families;
inspect saved parameters rather than attributing every difference solely to one
coefficient. The sklearn baseline uses liblinear, whose intercept regularization
also differs from this custom implementation.

CUSTOM MATHEMATICS 3 — RELIABILITY-WEIGHTED LOGIT POOLING
------------------------------------------------------
For raw per-series classifier logits l_s and lengths n_s:
    r_s = n_s / (n_s + tau),
    l_patient = sum_s r_s l_s / sum_s r_s,
    score = sigmoid(l_patient).

The old prior-relative expression simplifies exactly:
    l_prior + sum_s r_s(l_s-l_prior)/sum_s r_s
      = sum_s r_s l_s / sum_s r_s.
There is therefore no prior correction. The old Bayesian name has been removed.
For affine scaler/PCA/linear classification, averaging raw logits is equivalent
to classifying the identically weighted embedding mean. The implementation uses
that equivalent pooling at BOTH training and inference. It avoids the old train/
test aggregation mismatch and avoids inverse-logit clipping of probabilities.

At tau=0 all series have equal weight. As tau grows, relative preference for long
series increases; for very large tau the weights become approximately proportional
to n_s. Ties prefer SMALLER tau. Counts can themselves encode export confounding.
Duplicating all series equally preserves normalized weights; duplicating just one
series can change its influence. This is not an exact-deduplication algorithm.

NESTED SELECTION AND NUISANCE AUDIT: THE LEAKAGE BOUNDARY
------------------------------------------------------
For every outer split:
    outer-training patients T           held-out patients V
      -> inner candidate evaluation       never used for fitting/selection
      -> choose hyperparameters
      -> choose threshold on inner OOF
      -> fit predictor on all T
      -> predict disease score on V

For candidate selection:
1. Evaluate all predeclared hyperparameter settings on identical inner folds.
2. Retain settings within auc_tolerance (default 0.01) of best inner AUROC.
3. For residualization and BOTH custom logistic variants, optionally score
   nuisance predictability on this frontier with an extra training-local crossfit.
4. Prefer lower nonnegative nuisance prediction skill, within a fixed tolerance.
   Negative skill receives no extra reward. Use conservative parameter ties.
5. Baseline, control, and reliability grids use AUC plus parameter tie-breaking.

The extra inner nuisance evaluation is nested as follows:
    split T into A and B (inner training and validation)
    generate fixed-parameter OOF scores ONLY inside A
    fit nuisance(A) -> those scores
    predict the candidate's scores on B
All target-generating models exclude B, and the entire operation excludes V.
The additional targets hold the candidate parameters fixed; they do not perform
an unbounded recursive hyperparameter search.

For the FINAL outer nuisance diagnostic:
    chosen inner OOF scores on T -> fit nuisance(T) -> training scores
    apply this surrogate to nuisance(V)
    compare its prediction to the predictor's actual scores on V.
The surrogate never trains on global outer-OOF scores from other outer folds.
Those other target-generating models may have trained on V, the flaw fixed here.
Hyperparameters for these training targets were selected using T, which is
allowed; no outer-test patient influenced that selection.

The surrogate learns targets from smaller inner-trained models and is evaluated
against scores of a larger outer-trained model. This distribution mismatch and
small sample size can affect diagnostic R2. R2 is not a direct fraction of
confounding or a proof of independence. Scoring scale/calibration matters too.

R2, PREDICTIVE SKILL, AND REPEATED REPORTING
-----------------------------------------
Conventional score R2:
    1 - sum_i (s_i - predicted_s_i)^2 / sum_i (s_i - mean(s))^2.
Predictive score skill:
    1 - sum_i (s_i - predicted_s_i)^2 / sum_i (s_i - reference_i)^2,
where reference_i is the mean of training-only OOF targets in that outer fold.
The latter never learns its reference from the held-out score distribution.

For each nuisance block, component caps 3 and 5 are both evaluated and reported.
The largest configured cap is the primary diagnostic, not the one that looks
most favorable on test. This is a limited linear sensitivity analysis; nonlinear
nuisance predictability and omitted confounders may remain. Undefined constant-
target cases are reported as null, not fabricated finite R2 values.

Negative R2 means the surrogate predicts worse than its constant reference; it
does not mean negative dependence or that all confounding has been eliminated.
The familiar nuisance_score_r2 field now aliases the combined primary-cap R2,
but its training-local calculation is NOT numerically interchangeable with v1.

Every repeated split recomputes AUROC, AUPRC, all R2 probes, and selected settings.
Main-run metrics have explicit main_ prefixes in summary tables; repeated values
have repeated_ prefixes. Parameters and per-patient probe predictions are saved
for every seed, so lambda/rho/tau frequencies and failures can be inspected.
Paired deltas are comparison minus reference on matching seeds. A difference of
medians is not mislabeled as the median of paired differences. Repeats reuse the
same patients and cannot be treated as independent samples for significance.

DUPLICATES AND VALIDATION LIMITS
--------------------------------
Exact decoded-pixel hashes detect copied images across patients. By default such
cross-patient exact duplicates stop evaluation. Perceptual hashes and a BK-tree
screen visually similar pairs; similarity is not proof of identity. Manual
review is needed before treating candidate matches as duplicate components.

A patient bootstrap of fixed OOF predictions does not refit the entire training
and selection process; its interval understates some sources of model uncertainty.
No test here corrects all historical exploratory choices. No original DICOM
metadata, clinical covariates, external cohort, or expert segmentation truth is
invented. The outside support may contain heart tissue and real anatomy.
A high outside-control AUC is evidence of dataset structure, not proof that all
its signal is nonclinical. Penalizing a cardiac model's scores does not change the
AUC of a separately trained outside classifier automatically.

CS50 AI CONNECTION — CONCEPTUAL, NOT A CLAIM OF NEW INVENTION
-----------------------------------------------------------
Search: BK-tree metric search supports a complete pHash-radius audit.
Optimization: explicit objectives, gradients, and backtracking expose the math.
Minimax: the closed-form worst linear nuisance direction motivates the penalty;
this is not the game-tree minimax algorithm copied into a classifier.
Uncertainty: score fusion illustrates dependence and why multiplying correlated
series probabilities can overstate evidence. The corrected fusion is not Bayesian.
Learning: fixed pretrained representations plus supervised patient-level models.
Neural networks: pretrained segmenter/encoder are frozen, not trained on 30 labels.

Ridge projection, logistic regression, and covariance penalties are established
ideas adapted for this project. Their implementation is personal work, but no
claim of a newly discovered mathematical method is made. A convincing portfolio
emphasizes a testable hypothesis, transparent negative controls, rejected ideas,
implementation tests, and honest limitations instead of a chosen maximum AUC.

Possible FUTURE methods, not executed by this script:
* Simulated annealing over explicit reviewed sequence/view weights, with a
  training-only objective combining AUROC and nuisance dependence. Every search
  would have to remain nested and be evaluated on independent patients.
* Nuisance-matched nearest-neighbor diagnostics to test whether cardiac rankings
  remain informative among patients with similar acquisition/export features.
These are proposals, not completed experiments or validated improvements.

Primary references for concepts and API interpretation:
    https://cs50.harvard.edu/ai/notes/0/
    https://cs50.harvard.edu/ai/notes/2/
    https://cs50.harvard.edu/ai/notes/3/
    https://cs50.harvard.edu/ai/notes/4/
    https://cs50.harvard.edu/ai/notes/5/
    https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html
    https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html
    https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

HOW TO RUN: KAGGLE, COLAB, AND LOCAL PYTHON
-----------------------------------------
The file needs Python 3.10+ and compatible torch/torchvision, numpy, opencv-python,
scikit-learn, tqdm, and huggingface_hub. The ordinary segmenter path loads the
pinned TorchScript artifact and does not require importing MONAI Python itself.
First-time pretrained-weight loading needs network access or prepopulated caches.

main() has NO parameters and reads NO command-line arguments. Jupyter's injected
-f kernel.json is ignored. Code pasted into a cell is supported: __file__ is not
required; unavailable source path/hash is honestly recorded as null.

Edit the MAIN_* constants instead of adding CLI flags:
    MAIN_DATASET_PATH_OVERRIDE = None
    MAIN_OUTPUT_DIR_OVERRIDE = None
    MAIN_BATCH_SIZE = 8
    MAIN_REPEATED_CV_RUNS = 20
    MAIN_NUM_WORKERS = 0
    MAIN_FORCE_REBUILD_CACHE = False
    MAIN_USE_AMP = False
    MAIN_CHECK_PATHS_ONLY = False
    MAIN_SERIES_GROUPING = "patient_child"
    MAIN_SERIES_ANNOTATION_PATH = None
    MAIN_REGULARIZATION_NUISANCE_SCOPE = "export"
    MAIN_AUDIT_COMPONENT_CAPS = (3, 5)

Dataset override priority is source-code setting, environment variables
CAD_MRI_DATASET_PATH / CAD_DATASET_PATH, known Kaggle paths, bounded Kaggle search,
the historical local Windows path, then relative local candidates. A valid root
must contain both Normal and Sick. An explicitly invalid override fails rather
than silently selecting a different dataset.

Typical Kaggle root:
    /kaggle/input/datasets/danialsharifrazi/cad-cardiac-mri-dataset
Typical output root:
    /kaggle/working/cad_mri_university_showcase
Local dataset candidate:
    C:\F\_Develop\AI\Datasets\CAD Cardiac MRI Dataset

Run the file normally or paste it into a notebook cell. For a file already added
to a notebook's filesystem, %run /actual/path/cad_mri_university_showcase.py also
works. To check paths without model loading, set MAIN_CHECK_PATHS_ONLY=True.
Output must remain outside the input dataset to avoid contaminating discovery.

MAIN OUTPUTS
-------------
output_root/feature_cache/
    series_feature_bank_<fingerprint>.npz
    series_feature_bank_<fingerprint>.json (includes duplicate audit)
output_root/run_<timestamp>_<tag>/
    configuration.json
    run_metadata.json
    series_grouping_manifest.csv
    series_grouping_audit.json
    series_annotation_template.csv
    series_annotation_status.json
    nuisance_design.json
    duplicate_audit.json
    oof_<experiment>.csv (includes predicted nuisance scores and references)
    selected_params_<experiment>.json (all candidates and selected values)
    metrics_<experiment>.json
    repeated/seed_<seed>/ (the same OOF/parameter/metric files for every repeat)
    repeated_cv_results.csv
    selected_params_all_runs.csv
    selected_parameter_frequencies.json
    experiment_summary.csv
    paired_comparisons.json
    final_report.json

INTERPRETATION ORDER
--------------------
Start with patient/series counts and the grouping audit. Check annotation status
and duplicate findings. Compare cardiac versus support/shuffle/outside controls.
Then compare guarded versus matched no-guard custom logistic on the SAME seeds:
look jointly at paired AUROC and nuisance-R2 changes, parameter frequencies, and
optimizer iteration caps. Inspect both export and anatomy-proxy audits and both
component caps. A slight AUROC gain without reduced dependence is not successful
deconfounding; a large R2 drop accompanied by collapsed AUROC is not a useful
classifier. Finally require protocol-controlled and independent-cohort evidence
before making claims about disease specificity or generalization.

LOCAL TESTS VERSUS REAL DATA
---------------------------
The distributed regression tests check grouping, fingerprints, mathematical
identities, gradient correctness, patient isolation, notebook compatibility,
synthetic inference/cache contracts, and report serialization. Synthetic fixtures
and mocked pretrained networks do NOT measure real cardiac-MRI performance.
There is no hard-coded "validation passed" conclusion inside the model. Run the
tests, inspect the saved test report, and evaluate the new protocol on Kaggle.
"""

from __future__ import annotations
# Postpone evaluation of type annotations.  This keeps annotations readable and
# avoids importing or resolving every referenced type at function-definition time.

import csv
# Writes patient-level OOF predictions, repeated-CV results, and ranked summaries.

import hashlib
# Creates checkpoint digests, decoded-pixel hashes, and feature-cache identities.

import json
# Serializes configuration, software provenance, duplicate audits, and reports.

import math
# Provides numerically explicit operations such as gcd, square roots, and logic
# used by the affine permutation and custom optimization procedures.

import os
# Handles environment-aware path logic and portable folder traversal.

import platform
# Records the Python/runtime platform in run_metadata.json for reproducibility.

import random
# Seeds Python's pseudo-random generator alongside NumPy and PyTorch.

import time
# Measures model loading, extraction, evaluation, and total execution time.

from collections import defaultdict
# Accumulates per-series feature sums and duplicate-hash patient memberships.

from dataclasses import asdict, dataclass, replace
# Defines explicit data contracts and converts configuration to machine-readable JSON.

from pathlib import Path
# Provides safe, portable path construction for datasets, caches, and outputs.

from typing import Any, Literal, Sequence
# Makes tensor/data contracts and allowed argument values visible to reviewers.

import cv2
# Decodes grayscale images, resizes canvases, and computes the DCT pHash.

import numpy as np
# Implements array algebra, custom optimization, metrics preparation, and pooling.

import sklearn
# The package object is imported so its exact version can be written to metadata.

import torch
# Runs the frozen MONAI and EfficientNet models and batched image transformations.

import torch.nn as nn
# Supplies the Module abstraction used by the frozen EfficientNet wrapper.

import torch.nn.functional as F
# Supplies softmax, interpolation, and max-pooling used for segmentation support.

import torchvision
# The package object is imported to record its exact installed version.

from sklearn.decomposition import PCA
# Performs fold-local compact representations for cardiac and nuisance features.

from sklearn.linear_model import LogisticRegression, Ridge
# Logistic Regression is the small-cohort classifier; Ridge supports nuisance tests.

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
# All reported discrimination and threshold metrics are calculated at patient level.

from sklearn.model_selection import StratifiedKFold
# Creates outer and inner patient-level folds while preserving class proportions.

from sklearn.preprocessing import StandardScaler
# Fits feature scaling only on the current training partition.

from torch.utils.data import DataLoader, Dataset
# Decodes many images in deterministic batches without redefining the patient unit.

from torchvision import models
# Provides the explicitly selected ImageNet EfficientNet-B0 architecture and weights.

from tqdm import tqdm
# Displays progress during the expensive all-slice frozen feature extraction stage.


# =============================================================================
# 1. CONFIGURATION
# =============================================================================


# -----------------------------------------------------------------------------
# PORTABLE RUNTIME PATHS
# -----------------------------------------------------------------------------
#
# The full multi-experiment V6 suite used one simple environment split:
#
#     Kaggle canonical dataset path, when present
#         otherwise
#     a validated Windows development path.
#
# This compact portfolio keeps those two paths and extends them conservatively:
# source-code and environment overrides have priority, Kaggle input is searched
# only to a bounded depth, and local relative folders are checked beside both the
# script and the current working directory.  Detection is based on the actual
# data contract -- a root must contain both ``Normal/`` and ``Sick/`` -- rather
# than on a folder name alone.

DATASET_ENVIRONMENT_VARIABLES = (
    "CAD_MRI_DATASET_PATH",
    "CAD_DATASET_PATH",
)
# The first variable is the documented name.  The second is retained as a short
# compatibility alias for existing notebooks and local automation.

OUTPUT_ENVIRONMENT_VARIABLES = (
    "CAD_MRI_OUTPUT_DIR",
    "CAD_OUTPUT_DIR",
)

KAGGLE_DATASET_CANDIDATES = (
    Path("/kaggle/input/datasets/danialsharifrazi/cad-cardiac-mri-dataset"),
    Path("/kaggle/input/cad-cardiac-mri-dataset"),
)
# The first path matches the validated V6 configuration.  The second is Kaggle's
# common direct slug mount.  A bounded search below /kaggle/input is attempted
# afterward, so a harmless slug rename does not require source-code editing.

VALIDATED_LOCAL_DATASET_CANDIDATES = (
    Path(r"C:\F\_Develop\AI\Datasets\CAD Cardiac MRI Dataset"),
)
# This is the local Windows path used by the complete V6 suite.  It is treated as
# a candidate rather than an unconditional default, so Linux/macOS runs do not
# receive a misleading Windows path when the folder is absent.

COMMON_DATASET_DIRECTORY_NAMES = (
    "CAD Cardiac MRI Dataset",
    "cad-cardiac-mri-dataset",
)
# These names are searched only in a small set of predictable local locations.
# The code never scans an entire drive or home directory.

DEFAULT_KAGGLE_OUTPUT_DIRECTORY = Path(
    "/kaggle/working/cad_mri_university_showcase"
)
DEFAULT_LOCAL_OUTPUT_DIRECTORY_NAME = "cad_mri_university_showcase_outputs"

# -----------------------------------------------------------------------------
# MAIN EXECUTION SETTINGS — EDIT THESE INSTEAD OF PASSING COMMAND-LINE ARGUMENTS
# -----------------------------------------------------------------------------
# Jupyter, Colab, VS Code notebooks, and IPython kernels add their own arguments
# to ``sys.argv``.  A common example is:
#
#     -f /root/.local/share/jupyter/runtime/kernel-....json
#
# The previous command-line parser interpreted that kernel argument as a user
# error and stopped with ``SystemExit: 2``.  The entry point below never
# reads ``sys.argv``.  Runtime options are deliberately visible here as ordinary
# constants, which makes the same file safe in notebooks, Kaggle, and local Python.
#
# Leave path overrides as ``None`` for automatic Kaggle/local detection.  Set a
# string only when the dataset or output folder lives somewhere unusual.
MAIN_DATASET_PATH_OVERRIDE: str | None = None
MAIN_OUTPUT_DIR_OVERRIDE: str | None = None

MAIN_BATCH_SIZE = 8
MAIN_REPEATED_CV_RUNS = 20
MAIN_NUM_WORKERS = 0
MAIN_FORCE_REBUILD_CACHE = False
MAIN_USE_AMP = False
MAIN_CHECK_PATHS_ONLY = False
# New scientific settings remain explicit and notebook-safe; no CLI is parsed.
MAIN_SERIES_GROUPING = "patient_child"
MAIN_SERIES_ANNOTATION_PATH: str | None = None
MAIN_REGULARIZATION_NUISANCE_SCOPE = "export"
MAIN_AUDIT_COMPONENT_CAPS = (3, 5)
PIPELINE_REVISION = "2026-09-06-university-v2-training-local-audit"
FEATURE_CACHE_SCHEMA = "2026-09-06-series-membership-v2"

# Set MAIN_CHECK_PATHS_ONLY=True for a fast path diagnostic that loads no model.
# Notebook/Kaggle-cell safety: ``main()`` and run metadata never require
# ``__file__``.  When source code is executed directly from a cell, the script
# path/hash are recorded as unavailable instead of raising NameError.


@dataclass(frozen=True)
class ResolvedRuntimePaths:
    """Resolved input/output paths plus their provenance.

    Keeping the source strings is useful for reproducibility.  Two runs can use
    the same physical dataset while one was resolved automatically and the other
    was pinned through a source-code override.  The resolved absolute paths and
    their sources are saved in ``configuration.json`` through the main ``Config``
    dataclass.
    """

    dataset_path: Path
    output_dir: Path
    dataset_source: str
    output_source: str
    runtime_environment: str
    checked_dataset_paths: tuple[str, ...]


def _script_directory() -> Path:
    """Return the source-file directory, with a notebook-safe fallback.

    ``__file__`` exists when the pipeline is executed as a normal Python script
    or imported from a file.  Some notebook execution methods omit it, in which
    case the current working directory is the only reliable portable anchor.
    """

    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


def _runtime_source_identity() -> dict[str, str | None]:
    """Return notebook-safe source provenance without requiring ``__file__``.

    WHY THIS HELPER EXISTS
    ----------------------
    A normal ``python script.py`` execution defines ``__file__``.  Kaggle,
    Colab and Jupyter also allow the entire source to be pasted/executed as a
    notebook cell; in that execution mode ``__file__`` is not defined at all.
    Reading ``Path(__file__)`` directly would therefore raise ``NameError``
    *after* expensive setup, even though the scientific pipeline itself is
    otherwise valid.

    The source-file hash is reproducibility metadata, not a model input.  When
    a real source file exists we save its absolute path and SHA-256.  When code
    is running from an in-memory notebook cell, we record that fact explicitly
    and use ``None`` for the unavailable file path/hash.  No synthetic path or
    misleading checksum is invented.

    Returns
    -------
    dict
        ``script_path`` and ``script_sha256`` when a regular file is available,
        plus ``script_identity_source`` describing how provenance was resolved.
    """

    try:
        candidate = Path(__file__).resolve(strict=False)
    except NameError:
        return {
            "script_path": None,
            "script_sha256": None,
            "script_identity_source": "notebook_cell_without___file__",
        }

    if candidate.is_file():
        return {
            "script_path": str(candidate),
            "script_sha256": sha256_file(candidate),
            "script_identity_source": "__file__",
        }

    # Some interactive launchers may technically define ``__file__`` to a
    # pseudo-name that is not a real file.  Treat it as notebook-style source
    # rather than failing or hashing a nonexistent path.
    return {
        "script_path": str(candidate),
        "script_sha256": None,
        "script_identity_source": "__file___not_a_regular_file",
    }


def _is_kaggle_runtime() -> bool:
    """Identify Kaggle from its writable working directory contract."""

    return Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()


def _normalize_runtime_path(value: str | os.PathLike[str]) -> Path:
    """Expand environment/user markers and return a non-strict absolute path.

    ``strict=False`` allows output paths to be resolved before they exist.  The
    dataset resolver subsequently verifies existence and the Normal/Sick folder
    contract explicitly.
    """

    raw = str(value).strip()
    if not raw:
        raise ValueError("A runtime path cannot be empty.")
    expanded = os.path.expandvars(os.path.expanduser(raw))
    return Path(expanded).resolve(strict=False)


def _is_valid_dataset_root(path: Path) -> bool:
    """Return True only for a directory containing exact Normal/ and Sick/."""

    return (
        path.is_dir()
        and (path / "Normal").is_dir()
        and (path / "Sick").is_dir()
    )


def _bounded_directory_walk(
    base: Path,
    *,
    maximum_depth: int,
    maximum_directories: int = 500,
):
    """Yield directories breadth-first without performing an unbounded scan.

    The helper is used only around an explicitly supplied parent or below
    ``/kaggle/input``.  It stops after a fixed number of directories and never
    traverses symbolic-link directories, preventing accidental scans of a whole
    filesystem or cyclic links.
    """

    if maximum_depth < 0:
        raise ValueError("maximum_depth cannot be negative.")
    queue: list[tuple[Path, int]] = [(base, 0)]
    visited: set[str] = set()
    emitted = 0

    while queue and emitted < maximum_directories:
        current, depth = queue.pop(0)
        key = str(current)
        if key in visited:
            continue
        visited.add(key)
        if not current.is_dir():
            continue
        emitted += 1
        yield current

        if depth >= maximum_depth:
            continue
        try:
            children = sorted(
                (
                    child
                    for child in current.iterdir()
                    if child.is_dir()
                    and not child.is_symlink()
                    and not child.name.startswith(".")
                ),
                key=lambda child: child.name.casefold(),
            )
        except (OSError, PermissionError):
            continue
        queue.extend((child, depth + 1) for child in children)


def _find_dataset_root_from_candidate(
    candidate: Path,
    *,
    maximum_depth: int,
) -> Path | None:
    """Resolve a dataset root from a direct root or a nearby parent directory."""

    for possible_root in _bounded_directory_walk(
        candidate,
        maximum_depth=maximum_depth,
    ):
        if _is_valid_dataset_root(possible_root):
            return possible_root.resolve()
    return None


def _local_dataset_candidates() -> list[tuple[str, Path]]:
    """Return deterministic local candidates near cwd and the source file."""

    script_dir = _script_directory()
    cwd = Path.cwd().resolve()
    bases: list[tuple[str, Path]] = [
        ("current working directory", cwd),
        ("script directory", script_dir),
        ("current working directory/datasets", cwd / "datasets"),
        ("script directory/datasets", script_dir / "datasets"),
        ("script parent", script_dir.parent),
        ("script parent/datasets", script_dir.parent / "datasets"),
    ]

    candidates: list[tuple[str, Path]] = []
    seen: set[str] = set()
    for source, base in bases:
        direct_key = str(base)
        if direct_key not in seen:
            seen.add(direct_key)
            candidates.append((source, base))
        for folder_name in COMMON_DATASET_DIRECTORY_NAMES:
            path = base / folder_name
            key = str(path)
            if key not in seen:
                seen.add(key)
                candidates.append((f"{source}/{folder_name}", path))
    return candidates


def resolve_dataset_path(explicit_path: str | None = None) -> tuple[Path, str, tuple[str, ...]]:
    """Resolve the CAD MRI dataset root using a strict documented priority.

    Explicit source-code/environment values are authoritative: a typo raises a detailed
    error instead of falling through to a different dataset.  Automatic search
    uses exact known paths first and then bounded nearby scans.  The returned
    root is guaranteed to contain ``Normal/`` and ``Sick/``.
    """

    checked: list[str] = []

    def inspect(
        raw_path: str | os.PathLike[str],
        source: str,
        *,
        maximum_depth: int,
        strict: bool,
    ) -> tuple[Path, str] | None:
        """Check one prioritized candidate and enforce strict overrides."""

        candidate = _normalize_runtime_path(raw_path)
        checked.append(f"{source}: {candidate}")
        found = _find_dataset_root_from_candidate(
            candidate,
            maximum_depth=maximum_depth,
        )
        if found is not None:
            return found, source
        if strict:
            raise FileNotFoundError(
                "The explicitly selected dataset path is invalid.\n"
                f"Source: {source}\n"
                f"Path:   {candidate}\n"
                "Expected a directory containing both Normal/ and Sick/, or a "
                "nearby parent containing that dataset root."
            )
        return None

    if explicit_path is not None:
        result = inspect(
            explicit_path,
            "source-code MAIN_DATASET_PATH_OVERRIDE",
            maximum_depth=3,
            strict=True,
        )
        assert result is not None
        return result[0], result[1], tuple(checked)

    for variable_name in DATASET_ENVIRONMENT_VARIABLES:
        value = os.environ.get(variable_name)
        if value and value.strip():
            result = inspect(
                value,
                f"environment variable {variable_name}",
                maximum_depth=3,
                strict=True,
            )
            assert result is not None
            return result[0], result[1], tuple(checked)

    for candidate in KAGGLE_DATASET_CANDIDATES:
        result = inspect(
            candidate,
            f"known Kaggle path {candidate}",
            maximum_depth=1,
            strict=False,
        )
        if result is not None:
            return result[0], result[1], tuple(checked)

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        result = inspect(
            kaggle_input,
            "bounded Kaggle /kaggle/input search",
            maximum_depth=4,
            strict=False,
        )
        if result is not None:
            return result[0], result[1], tuple(checked)

    for candidate in VALIDATED_LOCAL_DATASET_CANDIDATES:
        result = inspect(
            candidate,
            f"validated V6 local path {candidate}",
            maximum_depth=1,
            strict=False,
        )
        if result is not None:
            return result[0], result[1], tuple(checked)

    for source, candidate in _local_dataset_candidates():
        result = inspect(
            candidate,
            f"automatic local candidate: {source}",
            maximum_depth=2,
            strict=False,
        )
        if result is not None:
            return result[0], result[1], tuple(checked)

    attempted = "\n".join(f"  - {entry}" for entry in checked)
    raise FileNotFoundError(
        "Could not locate the CAD Cardiac MRI dataset automatically.\n"
        "A valid root must contain both Normal/ and Sick/.\n\n"
        "Set one of the following:\n"
        "  MAIN_DATASET_PATH_OVERRIDE = '/path/to/dataset'\n"
        "  CAD_MRI_DATASET_PATH=/path/to/dataset\n\n"
        "Checked locations:\n"
        f"{attempted}"
    )


def resolve_output_directory(explicit_path: str | None = None) -> tuple[Path, str]:
    """Resolve a writable output destination without touching the dataset.

    Creation is deferred to ``run_research_portfolio``.  Keeping resolution and
    creation separate lets ``MAIN_CHECK_PATHS_ONLY`` inspect the decision without
    writing any file.
    """

    if explicit_path is not None:
        return (
            _normalize_runtime_path(explicit_path),
            "source-code MAIN_OUTPUT_DIR_OVERRIDE",
        )

    for variable_name in OUTPUT_ENVIRONMENT_VARIABLES:
        value = os.environ.get(variable_name)
        if value and value.strip():
            return (
                _normalize_runtime_path(value),
                f"environment variable {variable_name}",
            )

    if _is_kaggle_runtime():
        return (
            DEFAULT_KAGGLE_OUTPUT_DIRECTORY.resolve(strict=False),
            "automatic Kaggle /kaggle/working output",
        )

    script_dir = _script_directory()
    local_base = (
        script_dir
        if script_dir.is_dir() and os.access(script_dir, os.W_OK)
        else Path.cwd().resolve()
    )
    return (
        (local_base / DEFAULT_LOCAL_OUTPUT_DIRECTORY_NAME).resolve(strict=False),
        "automatic local output beside script"
        if local_base == script_dir
        else "automatic local output in current working directory",
    )


def resolve_runtime_paths(
    dataset_argument: str | None,
    output_argument: str | None,
) -> ResolvedRuntimePaths:
    """Resolve, validate, and describe all paths needed by one run."""

    dataset_path, dataset_source, checked = resolve_dataset_path(dataset_argument)
    output_dir, output_source = resolve_output_directory(output_argument)
    runtime_environment = "Kaggle" if _is_kaggle_runtime() else "local"
    return ResolvedRuntimePaths(
        dataset_path=dataset_path,
        output_dir=output_dir,
        dataset_source=dataset_source,
        output_source=output_source,
        runtime_environment=runtime_environment,
        checked_dataset_paths=checked,
    )


def print_resolved_runtime_paths(paths: ResolvedRuntimePaths) -> None:
    """Print one clear path-resolution summary before any expensive work."""

    print("\n" + "=" * 78, flush=True)
    print("PORTABLE RUNTIME PATHS", flush=True)
    print("=" * 78, flush=True)
    print(
        f"[PATHS] Runtime environment: {paths.runtime_environment}",
        flush=True,
    )
    print(f"[PATHS] Dataset source: {paths.dataset_source}", flush=True)
    print(f"[PATHS] Dataset root: {paths.dataset_path}", flush=True)
    print(f"[PATHS] Output source: {paths.output_source}", flush=True)
    print(f"[PATHS] Output directory: {paths.output_dir}", flush=True)
    print(
        "[PATHS] Dataset contract: Normal/ and Sick/ are present.",
        flush=True,
    )
    print("=" * 78, flush=True)


@dataclass(frozen=True)
class Config:
    """Immutable configuration for one complete research run.

    The fields are grouped by scientific role: reproducibility, geometry,
    segmentation, normalization, frozen checkpoints, nested evaluation, custom
    algorithms, caching, and duplicate auditing.  A configuration value may be
    chosen before a run or selected inside inner cross-validation, but it must
    never be adjusted after examining an outer-validation result.

    Serializing this dataclass makes it possible to reproduce exactly which
    assumptions created a feature bank and which assumptions affected only the
    downstream patient-level models.
    """

    dataset_path: str
    # Root containing the two class folders ``Normal`` and ``Sick``.

    output_dir: str
    # Run-specific destination for configuration, cache, OOF predictions, and reports.

    dataset_path_source: str = "provided programmatically"
    # Records whether the path came from a source-code override, an environment
    # variable, a known
    # Kaggle mount, the validated V6 Windows path, or automatic local discovery.

    output_dir_source: str = "provided programmatically"
    # Saved beside the resolved output path for reproducible runtime provenance.

    runtime_environment: str = "programmatic"
    series_grouping: str = "patient_child"
    # patient_child: all descendants of one immediate Directory_* child share
    # a proxy, as in the historical full-suite source. immediate_parent retains
    # the previous compact script's grouping for explicitly labelled sensitivity.
    series_annotations_path: str | None = None
    # Optional manually reviewed CSV. When supplied, use only explicit cardiac,
    # non-localizer, non-derived rows and sequence/view cells common to EVERY
    # patient. No patient is dropped and no sequence/view is inferred from paths.

    # Usually ``Kaggle`` or ``local`` when the notebook-safe entry point resolves it.

    # -------------------------------------------------------------------------
    # REPRODUCIBILITY AND HARDWARE
    # -------------------------------------------------------------------------
    # These values affect fold generation, batching, and numerical precision.
    # ``use_amp`` is opt-in because full precision is easier to reproduce and to
    # compare against the validated V6 audit.
    seed: int = 42
    batch_size: int = 8
    num_workers: int = 0
    use_amp: bool = False

    # -------------------------------------------------------------------------
    # GEOMETRY AND SEGMENTATION
    # -------------------------------------------------------------------------
    # MONAI receives a 256x256 canvas.  EfficientNet later receives a uniformly
    # resized 224x224 version of the complete aligned canvas.  The hard threshold,
    # plausibility gate, and dilation widths are fixed before model evaluation.
    monai_input_size: int = 256
    encoder_input_size: int = 224
    standardized_content_long_side: int = 240
    monai_hard_threshold: float = 0.50
    monai_min_area_ratio: float = 0.003
    monai_max_area_ratio: float = 0.50
    monai_min_peak_probability: float = 0.50
    monai_dilation_kernel: int = 31
    a17_extra_dilation_kernel: int = 15
    fallback_square_fraction: float = 0.65

    # -------------------------------------------------------------------------
    # REGION-ONLY ROBUST NORMALIZATION
    # -------------------------------------------------------------------------
    # Percentiles are estimated only from pixels visible in the final candidate
    # or control.  This prevents excluded heart pixels from setting the scale of
    # an outside-support control and vice versa.
    lower_percentile: float = 1.0
    upper_percentile: float = 99.0
    histogram_bins: int = 256
    minimum_visible_pixels: int = 64
    minimum_dynamic_range: float = 8.0 / 255.0

    # -------------------------------------------------------------------------
    # CONSERVATIVE LABEL-BLIND DARK-BORDER REMOVAL
    # -------------------------------------------------------------------------
    # A crop is accepted only for consecutive edge lines that are both dark and
    # nearly uniform, and only if a large central fraction remains.  Crop geometry
    # is retained as nuisance metadata rather than hidden from the audit.
    dark_line_max_mean: float = 12.0
    dark_line_max_std: float = 4.0
    dark_pixel_max_value: int = 20
    dark_pixel_min_fraction: float = 0.98
    max_crop_fraction_per_side: float = 0.20
    min_retained_fraction: float = 0.60
    min_padding_run: int = 2

    # -------------------------------------------------------------------------
    # FROZEN MODEL IDENTITIES
    # -------------------------------------------------------------------------
    # The MONAI repository revision and SHA-256 are pinned.  EfficientNet uses an
    # explicit weight enum rather than ``DEFAULT`` so a future torchvision release
    # cannot silently change the encoder checkpoint.
    monai_repo_id: str = "MONAI/ventricular_short_axis_3label"
    monai_revision: str = "eefc17c8e002cc8a567bbfce8f02d7d3116408f4"
    monai_filename: str = "models/model.ts"
    monai_sha256: str = (
        "27d5532401fa6c1883872fa21635adbb7615981e7f385d0c58dd75b355e340b3"
    )
    efficientnet_weights: str = "IMAGENET1K_V1"
    embedding_dim: int = 1280

    # -------------------------------------------------------------------------
    # NESTED PATIENT-LEVEL EVALUATION
    # -------------------------------------------------------------------------
    # Hyperparameters and thresholds are selected in inner folds.  Outer folds
    # receive one score per Directory_* patient.  Repeated runs change only the
    # split seed and are sensitivity analyses, not independent cohorts.
    outer_folds: int = 5
    inner_folds: int = 3
    repeated_cv_runs: int = 20
    c_grid: tuple[float, ...] = (0.01, 0.1, 1.0, 10.0)
    pca_variance: float = 0.95
    max_pca_components: int = 10
    auc_tolerance: float = 0.01
    bootstrap_replicates: int = 2000

    # -------------------------------------------------------------------------
    # CUSTOM METHOD 1: FOLD-LOCAL NUISANCE PROJECTION
    # -------------------------------------------------------------------------
    # Ridge alpha shrinks the nuisance regression; rho controls the fraction
    # subtracted. Scaling/PCA and coefficients are training-only. Grid selection
    # uses training labels/score diagnostics, never outer-test metrics.
    residual_alpha_grid: tuple[float, ...] = (1.0, 10.0)
    residual_rho_grid: tuple[float, ...] = (0.0, 0.25, 0.5, 1.0)
    # X_clean = X_centered - rho * Z_encoded @ B. rho=0 is the no-removal
    # candidate; rho=1 is full residualization. Small grids limit adaptive search.
    regularization_nuisance_scope: str = "export"
    # export excludes control-image embeddings and MONAI geometry. combined and
    # anatomy_proxy are explicit sensitivity settings, NOT automatic choices.
    use_two_objective_selection: bool = True
    nuisance_skill_tolerance: float = 0.01
    # For residualization and BOTH custom logistic variants, among candidates
    # within auc_tolerance of best inner AUROC prefer lower nonnegative nuisance
    # prediction skill measured with an additional training-local crossfit. Negative skill is treated as zero, not as extra success.
    audit_pca_components: tuple[int, ...] = (3, 5)
    audit_ridge_alpha: float = 1.0
    # Both caps are reported, never chosen on outer-test performance. The largest
    # cap is the predeclared primary score, the smaller cap a sensitivity probe.

    max_nuisance_components: int = 5
    nuisance_random_projection_dim: int = 16
    nuisance_random_projection_seed: int = 91_731

    # -------------------------------------------------------------------------
    # CUSTOM METHOD 2: WORST-CASE LINEAR CONFOUNDER GUARD
    # -------------------------------------------------------------------------
    # ``guard_lambda_grid`` controls the adversarial covariance penalty and
    # ``guard_l2_grid`` controls ordinary weight shrinkage.  Optimization uses
    # explicit gradients with backtracking line search.
    guard_lambda_grid: tuple[float, ...] = (0.1, 1.0, 10.0)
    guard_l2_grid: tuple[float, ...] = (0.01, 0.1, 1.0)
    guard_learning_rate: float = 0.20
    guard_max_iterations: int = 1500
    guard_tolerance: float = 1e-7

    # -------------------------------------------------------------------------
    # CUSTOM METHOD 3: RELIABILITY-WEIGHTED LOGIT FUSION (NOT BAYESIAN)
    # -------------------------------------------------------------------------
    # r_s=n_s/(n_s+tau); average raw decision logits without probability clipping.
    # tau=0 gives equal series weights. Larger tau increases RELATIVE preference
    # for longer series. Tie-breaking therefore prefers SMALLER tau.
    fusion_tau_grid: tuple[float, ...] = (0.0, 5.0, 20.0)


    # -------------------------------------------------------------------------
    # CACHE AND DUPLICATE AUDITS
    # -------------------------------------------------------------------------
    # The cache stores one mean embedding per series proxy.  Exact cross-patient
    # pixel duplicates can stop evaluation; pHash matches remain review candidates.
    force_rebuild_cache: bool = False
    fail_on_cross_patient_exact_duplicate: bool = True
    run_perceptual_hash_audit: bool = True
    phash_hamming_radius: int = 3


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
# File extensions accepted during discovery.  The public release is JPEG-based,
# but allowing common raster formats makes local testing easier without changing
# the patient/series contract.

FEATURE_MODES = (
    "cardiac",
    "support_only",
    "shuffled_intensity",
    "outside_support",
)
# The four views are deliberately small and mutually interpretable.  They are
# encoded by the same frozen network and aggregated by the same series/patient
# rules, so comparisons focus on visible information rather than model capacity.

METADATA_NAMES = (
    "native_height",
    "native_width",
    "native_aspect_ratio",
    "file_bytes_per_pixel",
    "crop_fraction",
    "content_fraction",
    "monai_valid",
    "support_fraction",
)
# Metadata columns are averaged within series and patients, then appended to the
# nuisance matrix.  They never enter the ordinary baseline classifier directly.

EXPERIMENTS = (
    "BASELINE_A17",
    "CONTROL_SUPPORT_ONLY",
    "CONTROL_SHUFFLED_INTENSITY",
    "CONTROL_OUTSIDE_SUPPORT",
    "CUSTOM_NUISANCE_PROJECTION",
    "CUSTOM_LOGISTIC_NO_GUARD",
    "CUSTOM_MINIMAX_CONFOUNDER_GUARD",
    "CUSTOM_RELIABILITY_WEIGHTED_LOGIT_FUSION",
)
# Experiment order is fixed before evaluation.  It also determines presentation
# order before the final table is re-ranked by repeated-CV median AUROC.


@dataclass(frozen=True)
class SliceRecord:
    """Immutable index entry for one decoded-image candidate.

    ``patient_id`` is always the validated ``Directory_*`` unit.  ``series_id``
    is only a patient-scoped folder proxy.  The class label is carried as
    metadata for later patient-level evaluation and is never passed to MONAI,
    EfficientNet, border detection, hashing, or control construction.
    """

    path: str
    label: int
    patient_id: str
    series_id: str


@dataclass
class SeriesBank:
    """Compact frozen representation with one row per series proxy.

    ``features[mode]`` has shape ``[n_series, 1280]`` and stores the exact mean
    of all frozen slice embeddings in that series.  ``metadata`` stores the
    corresponding mean export/QC variables, while ``n_slices`` preserves the
    slice counts available to reliability-weighted fusion.  Array rows are kept
    in a single deterministic ``series_ids`` order.
    """

    series_ids: np.ndarray
    patient_ids: np.ndarray
    labels: np.ndarray
    n_slices: np.ndarray
    features: dict[str, np.ndarray]
    metadata: np.ndarray
    pooling_cells: np.ndarray | None = None
    # Populated ONLY by the optional manual annotation stage, not the raw cache.


@dataclass
class PatientBank:
    """One-row-per-patient representation used for supervised learning.

    Each image feature mode is averaged equally across a patient's series
    proxies, so a long exported series cannot dominate merely by containing more
    JPEG frames.  ``nuisance`` combines compressed support-only, outside-support,
    and acquisition/export variables.  ``patient_to_series_rows`` allows the
    reliability-weighted method to return to series-level pooling without redefining
    a series as an independent labeled patient.
    """

    patient_ids: np.ndarray
    labels: np.ndarray
    features: dict[str, np.ndarray]
    nuisance: np.ndarray
    metadata: np.ndarray
    patient_to_series_rows: dict[str, np.ndarray]
    nuisance_blocks: dict[str, np.ndarray] | None = None
    # Separate export, anatomy_proxy, and combined diagnostics. None is allowed
    # only for compatibility with programmatically built legacy banks.


def validate_config(config: Config) -> None:
    """Validate every predeclared numerical and structural contract early.

    This function runs before dataset scanning, network downloads, or GPU work.
    It checks odd dilation kernels, legal probability/percentile intervals,
    positive hyperparameter grids, valid fold counts, and checkpoint-digest
    formatting.  Failing early prevents an expensive partial cache from being
    mistaken for a scientifically valid run.
    """

    dataset_root = _normalize_runtime_path(config.dataset_path)
    output_root = _normalize_runtime_path(config.output_dir)
    if not _is_valid_dataset_root(dataset_root):
        raise FileNotFoundError(
            "config.dataset_path must contain both Normal/ and Sick/: "
            f"{dataset_root}"
        )
    if output_root.exists() and not output_root.is_dir():
        raise NotADirectoryError(
            f"config.output_dir exists but is not a directory: {output_root}"
        )
    try:
        output_root.relative_to(dataset_root)
    except ValueError:
        pass
    else:
        raise ValueError(
            "The output directory must not be inside the dataset root. "
            "Otherwise generated files could contaminate future discovery or "
            f"dataset fingerprints: {output_root}"
        )
    if config.batch_size < 1:
        raise ValueError("batch_size must be at least one.")
    if config.num_workers < 0:
        raise ValueError("num_workers cannot be negative.")

    for name, value in (
        ("monai_dilation_kernel", config.monai_dilation_kernel),
        ("a17_extra_dilation_kernel", config.a17_extra_dilation_kernel),
    ):
        if value <= 0 or value % 2 == 0:
            raise ValueError(f"{name} must be a positive odd integer.")
    if not 0.0 < config.fallback_square_fraction <= 1.0:
        raise ValueError("fallback_square_fraction must lie in (0, 1].")
    if not 0.0 <= config.lower_percentile < config.upper_percentile <= 100.0:
        raise ValueError("Invalid robust-normalization percentile interval.")
    if config.histogram_bins < 2 or config.minimum_visible_pixels < 1:
        raise ValueError("Histogram bins and visible-pixel minimum must be positive.")
    if config.minimum_dynamic_range <= 0.0:
        raise ValueError("minimum_dynamic_range must be positive.")
    if config.outer_folds < 2 or config.inner_folds < 2:
        raise ValueError("Outer and inner CV need at least two folds.")
    if config.repeated_cv_runs < 1:
        raise ValueError("repeated_cv_runs must be at least one.")
    if not config.c_grid or any(value <= 0 for value in config.c_grid):
        raise ValueError("Every Logistic Regression C must be positive.")
    if not config.residual_alpha_grid or any(
        value <= 0 for value in config.residual_alpha_grid
    ):
        raise ValueError("Every residualization alpha must be positive.")
    if not config.guard_lambda_grid or any(
        value <= 0 for value in config.guard_lambda_grid
    ):
        raise ValueError("Guard penalties must be positive; lambda=0 has its own ablation.")
    if not config.guard_l2_grid or any(value <= 0 for value in config.guard_l2_grid):
        raise ValueError("Every custom-model L2 value must be positive.")
    if not config.fusion_tau_grid or any(value < 0 for value in config.fusion_tau_grid):
        raise ValueError("Reliability tau must be nonnegative; tau=0 is equal pooling.")
    if not config.residual_rho_grid or any(not 0 <= x <= 1 for x in config.residual_rho_grid):
        raise ValueError("Residualization rho must lie in [0, 1].")
    if config.series_grouping not in {"patient_child", "immediate_parent"}:
        raise ValueError("Unknown series_grouping policy.")
    if config.regularization_nuisance_scope not in {"export", "anatomy_proxy", "combined"}:
        raise ValueError("Unknown regularization_nuisance_scope.")
    if (not config.audit_pca_components or any(x < 1 for x in config.audit_pca_components)
            or len(set(config.audit_pca_components)) != len(config.audit_pca_components)):
        raise ValueError("Audit component caps must be distinct positive integers.")
    if config.audit_ridge_alpha <= 0 or config.nuisance_skill_tolerance < 0:
        raise ValueError("Invalid nuisance audit regularization/tolerance.")
    if not 0 <= config.auc_tolerance <= 1:
        raise ValueError("auc_tolerance must lie in [0, 1].")
    if config.series_annotations_path and not Path(config.series_annotations_path).is_file():
        raise FileNotFoundError(config.series_annotations_path)

    if config.nuisance_random_projection_dim < 1:
        raise ValueError("nuisance_random_projection_dim must be positive.")
    if len(config.monai_sha256) != 64:
        raise ValueError("monai_sha256 must contain 64 hexadecimal characters.")


# =============================================================================
# 2. GENERAL UTILITIES
# =============================================================================


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch random generators.

    The seed controls fold shuffling, bootstrap resampling, fixed random nuisance
    projections, and deterministic initialization of custom optimization.  The
    function deliberately does not promise bit-identical results across all GPU,
    CUDA, driver, and library combinations; software versions are recorded so
    residual numerical variation can be audited.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def write_json(path: Path, value: Any) -> None:
    """Write a stable, human-readable JSON artifact.

    Parent folders are created automatically.  Keys are sorted and indentation
    is fixed so configuration and result changes are easy to inspect with a
    version-control diff.  Only JSON-serializable values should reach this helper.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True), encoding="utf-8")


def stable_sigmoid(x: np.ndarray) -> np.ndarray:
    """Evaluate the logistic function without exponential overflow.

    Positive and negative inputs use algebraically equivalent branches.  This
    avoids evaluating ``exp(-x)`` for very negative ``x`` or ``exp(x)`` for very
    positive ``x`` and is used by the custom optimizer and logit fusion.
    """

    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def logit(p: np.ndarray | float, eps: float = 1e-6) -> np.ndarray:
    """Convert probabilities to log-odds after fixed numerical clipping.

    Clipping to ``[eps, 1-eps]`` prevents infinite evidence when a linear model
    produces a score numerically equal to zero or one.  The clipping constant is
    numerical protection, not a learned calibration parameter.
    """
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def sha256_file(path: Path) -> str:
    """Return the SHA-256 digest of a file using bounded-memory streaming.

    The helper verifies the pinned MONAI artifact and records source integrity.
    Reading in one-megabyte chunks avoids loading a model checkpoint entirely
    into RAM.
    """
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def configuration_fingerprint(config: Config, records: Sequence[SliceRecord]) -> str:
    """Hash feature preprocessing AND each image's exact patient/series membership.

    Classifier grids, nuisance audit, annotations (applied after caching), and
    repeat counts do not affect this key. The schema and grouping policy do.
    A changed policy intentionally invalidates the old 2,879-series cache; no
    target count is forced and no old cache is renamed to pretend compatibility.
    """
    feature_fields = (
        "batch_size", "use_amp", "monai_input_size", "encoder_input_size",
        "standardized_content_long_side", "monai_hard_threshold", "monai_min_area_ratio",
        "monai_max_area_ratio", "monai_min_peak_probability", "monai_dilation_kernel",
        "a17_extra_dilation_kernel", "fallback_square_fraction", "lower_percentile",
        "upper_percentile", "histogram_bins", "minimum_visible_pixels",
        "minimum_dynamic_range", "dark_line_max_mean", "dark_line_max_std",
        "dark_pixel_max_value", "dark_pixel_min_fraction", "max_crop_fraction_per_side",
        "min_retained_fraction", "min_padding_run", "monai_repo_id", "monai_revision",
        "monai_filename", "monai_sha256", "efficientnet_weights", "embedding_dim",
        "series_grouping",
    )
    identity = {name: getattr(config, name) for name in feature_fields}
    identity.update(schema=FEATURE_CACHE_SCHEMA, feature_modes=FEATURE_MODES,
                    torch=torch.__version__, torchvision=torchvision.__version__,
                    run_perceptual_hash_audit=config.run_perceptual_hash_audit,
                    phash_hamming_radius=config.phash_hamming_radius,
                    device="cuda" if torch.cuda.is_available() else "cpu")
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode("utf-8"))
    root = Path(config.dataset_path).resolve()
    for record in records:
        path = Path(record.path)
        stat = path.stat()
        row = [path.resolve().relative_to(root).as_posix(), record.patient_id,
               record.series_id, int(record.label), stat.st_size, stat.st_mtime_ns]
        digest.update(json.dumps(row, separators=(",", ":")).encode("utf-8"))
    return digest.hexdigest()[:16]


# =============================================================================
# 3. DATASET DISCOVERY
# =============================================================================


def series_id_for_image(image_path: Path, patient_dir: Path, grouping: str) -> str:
    """Map one descendant path without interpreting folder names as MR sequences.

    patient_child reproduces the available full-suite rule: e.g.
    Directory_1/SR_1/subfolder/im.jpg -> Directory_1/SR_1. Images directly under
    the patient become Directory_1/__ROOT__. immediate_parent preserves the old
    compact mapping (including ROOT instead of __ROOT__) for audit only.
    """
    parts = image_path.relative_to(patient_dir).parts[:-1]
    if grouping == "patient_child":
        suffix = parts[0] if parts else "__ROOT__"
    elif grouping == "immediate_parent":
        suffix = "/".join(parts) if parts else "ROOT"
    else:
        raise ValueError(f"Unknown grouping: {grouping}")
    return f"{patient_dir.name}/{suffix}"


def write_series_grouping_audit(records: Sequence[SliceRecord], config: Config) -> dict[str, Any]:
    """Save a per-image mapping plus per-patient counts under BOTH grouping rules.

    This diagnoses the reported 2,859-versus-2,879 discrepancy without asserting
    its cause or coercing the current cohort to either count. File paths and
    grouping choices are audit data, never classifier input. Counts are measured
    from the mounted dataset, not copied from a historical console log.
    """
    root = Path(config.dataset_path).resolve()
    output = Path(config.output_dir)
    groups: dict[str, dict[str, set[str]]] = defaultdict(lambda: defaultdict(set))
    changed = 0
    with (output / "series_grouping_manifest.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["relative_path", "patient_id", "selected_series_id",
                         "patient_child_series_id", "previous_immediate_parent_series_id"])
        for record in records:
            relative = Path(record.path).resolve().relative_to(root)
            patient_dir = root / relative.parts[0] / record.patient_id
            child = series_id_for_image(Path(record.path), patient_dir, "patient_child")
            immediate = series_id_for_image(Path(record.path), patient_dir, "immediate_parent")
            groups[record.patient_id]["patient_child"].add(child)
            groups[record.patient_id]["immediate_parent"].add(immediate)
            changed += int(child != immediate)
            writer.writerow([relative.as_posix(), record.patient_id, record.series_id, child, immediate])
    rows = [{"patient_id": p, "patient_child_series": len(g["patient_child"]),
             "immediate_parent_series": len(g["immediate_parent"])} for p, g in sorted(groups.items())]
    summary = {"selected_policy": config.series_grouping, "patients": len(rows),
               "images": len(records), "images_with_changed_series_id": changed,
               "patient_child_series": sum(r["patient_child_series"] for r in rows),
               "immediate_parent_series": sum(r["immediate_parent_series"] for r in rows),
               "patients_detail": rows,
               "historical_reported_series_counts": [2859, 2879],
               "historical_counts_are_not_forced": True}
    write_json(output / "series_grouping_audit.json", summary)
    print(f"[SERIES AUDIT] patient_child={summary['patient_child_series']}; "
          f"immediate_parent={summary['immediate_parent_series']}; selected={config.series_grouping}", flush=True)
    return summary


def discover_records(dataset_path: str, grouping: str = "patient_child") -> list[SliceRecord]:
    """Build the complete image manifest without decoding pixels.

    Hard invariants:
    * ``Directory_*`` is the patient unit;
    * every descendant image remains assigned to that patient;
    * patient_child groups ALL descendants of one immediate patient child;
    * immediate_parent is an explicit previous-version sensitivity policy;
    * one patient may not occur under both labels;
    * both classes must be present.

    The deterministic final sort fixes the row order used by caching and audit
    operations.
    """

    root = Path(dataset_path).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {root}")

    records: list[SliceRecord] = []
    patient_to_label: dict[str, int] = {}
    for class_name, label in (("Normal", 0), ("Sick", 1)):
        class_dir = root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")

        patient_dirs = sorted(
            path
            for path in class_dir.glob("Directory_*")
            if path.is_dir()
        )
        if not patient_dirs:
            raise RuntimeError(f"No Directory_* folders found under {class_dir}")

        for patient_dir in patient_dirs:
            patient_id = patient_dir.name
            previous = patient_to_label.get(patient_id)
            if previous is not None and previous != label:
                raise RuntimeError(
                    f"Patient {patient_id} appears under both class labels."
                )
            patient_to_label[patient_id] = label

            image_paths = sorted(
                path
                for path in patient_dir.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )
            if not image_paths:
                raise RuntimeError(f"Patient {patient_id} has no image files.")

            for image_path in image_paths:
                series_id = series_id_for_image(image_path, patient_dir, grouping)
                records.append(
                    SliceRecord(
                        path=str(image_path),
                        label=label,
                        patient_id=patient_id,
                        series_id=series_id,
                    )
                )

    records.sort(key=lambda row: (row.patient_id, row.series_id, row.path))
    labels = [patient_to_label[patient] for patient in sorted(patient_to_label)]
    if len(set(labels)) != 2:
        raise RuntimeError("Both Normal and Sick patients are required.")

    series_count = len({row.series_id for row in records})
    print(
        "[DATA] "
        f"patients={len(patient_to_label)}, "
        f"Normal={sum(v == 0 for v in patient_to_label.values())}, "
        f"Sick={sum(v == 1 for v in patient_to_label.values())}, "
        f"images={len(records)}, series_proxies={series_count}",
        flush=True,
    )
    return records


# =============================================================================
# 4. LABEL-BLIND IMAGE STANDARDIZATION
# =============================================================================


def _line_is_dark_and_uniform(line: np.ndarray, config: Config) -> bool:
    """Test whether one edge row or column resembles removable padding.

    A line must simultaneously have low mean intensity, low standard deviation,
    and a high fraction of pixels below a dark-value threshold.  The rule is
    fixed and label-blind; it never reads patient class, MONAI output, or model
    score.
    """
    line = line.astype(np.float32)
    return bool(
        float(line.mean()) <= config.dark_line_max_mean
        and float(line.std()) <= config.dark_line_max_std
        and float(np.mean(line <= config.dark_pixel_max_value))
        >= config.dark_pixel_min_fraction
    )


def _count_dark_edge_lines(
    image: np.ndarray,
    axis: Literal[0, 1],
    from_end: bool,
    config: Config,
) -> int:
    """Count consecutive padding-like lines from one selected image edge.

    Counting stops at the first non-padding-like line and is capped by a fixed
    fraction of the native dimension.  A run shorter than ``min_padding_run`` is
    ignored, reducing the risk that isolated dark anatomy triggers a crop.
    """

    length = image.shape[axis]
    maximum = int(math.floor(length * config.max_crop_fraction_per_side))
    count = 0
    indices = range(length - 1, -1, -1) if from_end else range(length)
    for index in indices:
        line = image[index, :] if axis == 0 else image[:, index]
        if not _line_is_dark_and_uniform(line, config):
            break
        count += 1
        if count >= maximum:
            break
    return count if count >= config.min_padding_run else 0


def remove_conservative_dark_border(
    image: np.ndarray, config: Config
) -> tuple[np.ndarray, float]:
    """Remove only safe consecutive dark and nearly uniform edge lines.

    Candidate crop widths are detected independently on all four sides.  A
    second safety check rejects the crop if too little native height or width
    would remain.  The returned fraction is recorded as nuisance/QC metadata so
    the project can test whether preprocessing geometry itself predicts class.
    """

    height, width = image.shape
    top = _count_dark_edge_lines(image, axis=0, from_end=False, config=config)
    bottom = _count_dark_edge_lines(image, axis=0, from_end=True, config=config)
    left = _count_dark_edge_lines(image, axis=1, from_end=False, config=config)
    right = _count_dark_edge_lines(image, axis=1, from_end=True, config=config)

    retained_height = height - top - bottom
    retained_width = width - left - right
    if (
        retained_height < config.min_retained_fraction * height
        or retained_width < config.min_retained_fraction * width
    ):
        top = bottom = left = right = 0
        retained_height, retained_width = height, width

    cropped = image[top : height - bottom, left : width - right]
    crop_fraction = 1.0 - (cropped.size / float(image.size))
    return cropped, float(crop_fraction)


def resize_to_fixed_canvas(
    image: np.ndarray,
    content_long_side: int,
    canvas_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Resize retained content with preserved aspect ratio into one square canvas.

    The longest retained side becomes ``content_long_side`` and the result is
    centered in a ``canvas_size`` square.  A second binary canvas marks true
    resized content versus zero padding.  Raw and MONAI intensity views call
    this same geometric rule and must produce identical content masks.
    """

    height, width = image.shape
    scale = content_long_side / float(max(height, width))
    new_height = max(1, int(round(height * scale)))
    new_width = max(1, int(round(width * scale)))
    interpolation = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (new_width, new_height), interpolation=interpolation)

    canvas = np.zeros((canvas_size, canvas_size), dtype=np.float32)
    content = np.zeros((canvas_size, canvas_size), dtype=np.float32)
    top = (canvas_size - new_height) // 2
    left = (canvas_size - new_width) // 2
    canvas[top : top + new_height, left : left + new_width] = resized
    content[top : top + new_height, left : left + new_width] = 1.0
    return canvas, content


def prepare_slice(
    path: str, config: Config
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Decode one image and construct all label-independent slice inputs.

    Processing order:
    1. decode grayscale pixels;
    2. remove only conservative dark border runs;
    3. create raw uint8/255 and MONAI min-max intensity views;
    4. place both on identical fixed canvases;
    5. compute export/QC metadata;
    6. compute exact decoded-pixel and perceptual hashes.

    The returned label-free tensors are later batched by ``SliceDataset``.  An
    unreadable file raises an error instead of being silently omitted.
    """

    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise RuntimeError(f"OpenCV could not decode: {path}")
    native_height, native_width = image.shape
    cropped, crop_fraction = remove_conservative_dark_border(image, config)

    raw_01 = cropped.astype(np.float32) / 255.0
    minimum = float(cropped.min())
    maximum = float(cropped.max())
    if maximum > minimum:
        monai_01 = (cropped.astype(np.float32) - minimum) / (maximum - minimum)
    else:
        monai_01 = np.zeros_like(cropped, dtype=np.float32)

    raw_canvas, content_mask = resize_to_fixed_canvas(
        raw_01,
        config.standardized_content_long_side,
        config.monai_input_size,
    )
    monai_canvas, second_content_mask = resize_to_fixed_canvas(
        monai_01,
        config.standardized_content_long_side,
        config.monai_input_size,
    )
    if not np.array_equal(content_mask, second_content_mask):
        raise RuntimeError("Raw and MONAI canvas geometries are not aligned.")

    file_bytes = Path(path).stat().st_size
    pixel_hash = hashlib.sha256(image.tobytes()).hexdigest()
    metadata = np.asarray(
        [
            float(native_height),
            float(native_width),
            float(native_width / max(1, native_height)),
            float(file_bytes / max(1, image.size)),
            float(crop_fraction),
            float(content_mask.mean()),
        ],
        dtype=np.float32,
    )
    return (
        raw_canvas,
        monai_canvas,
        content_mask,
        metadata,
        pixel_hash,
        perceptual_hash_64(image),
    )


def perceptual_hash_64(image: np.ndarray) -> int:
    """Compute a 64-bit DCT perceptual hash for screening near duplicates.

    The image is reduced to a low-frequency 8x8 DCT block and thresholded around
    its non-DC median.  Hamming proximity is only a candidate signal; visually
    similar anatomy, blank frames, or repeated acquisition patterns can collide.
    Exact duplication decisions therefore require decoded-pixel hashes or manual
    review.
    """

    resized = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(resized.astype(np.float32))[:8, :8]
    values = dct.flatten()
    median = float(np.median(values[1:]))
    bits = values > median
    result = 0
    for bit in bits:
        result = (result << 1) | int(bit)
    return int(result)


class SliceDataset(Dataset):
    """PyTorch dataset that decodes and standardizes one slice per request.

    The dataset returns aligned tensors and immutable identifiers.  It does not
    perform augmentation and does not expose labels to frozen image models.  A
    deterministic DataLoader order allows series sums, hashes, and cache rows to
    be reconstructed exactly.
    """

    def __init__(self, records: Sequence[SliceRecord], config: Config):
        """Store a deterministic record list and immutable run configuration.

        Records are copied into a list so later iteration is unaffected by an
        external generator or mutation of the caller's sequence.
        """
        self.records = list(records)
        self.config = config

    def __len__(self) -> int:
        """Return the number of image rows in the discovered manifest."""
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        """Decode one record and return aligned tensors plus audit metadata.

        The dictionary format lets the DataLoader collate numeric tensors while
        preserving patient IDs, series IDs, exact hashes, and pHash strings.  The
        patient label remains metadata and is not used to construct any image view.
        """
        record = self.records[index]
        raw, monai, content, metadata, pixel_hash, phash = prepare_slice(
            record.path, self.config
        )
        return {
            "raw": torch.from_numpy(raw).unsqueeze(0),
            "monai": torch.from_numpy(monai).unsqueeze(0),
            "content": torch.from_numpy(content).unsqueeze(0),
            "metadata": torch.from_numpy(metadata),
            "label": torch.tensor(record.label, dtype=torch.long),
            "patient_id": record.patient_id,
            "series_id": record.series_id,
            "pixel_hash": pixel_hash,
            "phash": f"{phash:016x}",
        }


# =============================================================================
# 5. FROZEN MONAI AND EFFICIENTNET MODELS
# =============================================================================


def load_monai_segmenter(config: Config, device: torch.device) -> torch.jit.ScriptModule:
    """Download, verify, load, and sanity-check the pinned MONAI model.

    The Hugging Face repository is pinned to an immutable revision.  SHA-256 is
    checked before ``torch.jit.load``.  A zero-input inference verifies that the
    runtime output can be normalized to the expected four-channel shape.  This
    preferred TorchScript path avoids importing or rebuilding the full MONAI
    network during ordinary runs.
    """

    try:
        from huggingface_hub import hf_hub_download
    except ImportError as error:
        raise RuntimeError(
            "Install huggingface_hub before running feature extraction."
        ) from error

    model_path = Path(
        hf_hub_download(
            repo_id=config.monai_repo_id,
            filename=config.monai_filename,
            revision=config.monai_revision,
        )
    )
    actual_digest = sha256_file(model_path)
    if actual_digest.lower() != config.monai_sha256.lower():
        raise RuntimeError(
            "MONAI TorchScript SHA-256 mismatch. "
            f"Expected {config.monai_sha256}, received {actual_digest}."
        )

    model = torch.jit.load(str(model_path), map_location=device).eval()
    with torch.inference_mode():
        output = model(torch.zeros(1, 1, config.monai_input_size, config.monai_input_size, device=device))
    output = normalize_monai_output(output)
    if output.shape != (1, 4, config.monai_input_size, config.monai_input_size):
        raise RuntimeError(f"Unexpected MONAI output shape: {tuple(output.shape)}")
    return model


def normalize_monai_output(output: Any) -> torch.Tensor:
    """Extract a tensor from the supported TorchScript output containers.

    Different export/runtime combinations may return a tensor directly, a
    one-element sequence, or a dictionary containing ``pred``, ``logits``, or
    ``output``.  Recursive normalization keeps the downstream segmentation code
    independent of that shallow container choice while rejecting unknown types.
    """

    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (tuple, list)) and output:
        return normalize_monai_output(output[0])
    if isinstance(output, dict):
        for key in ("pred", "logits", "output"):
            if key in output:
                return normalize_monai_output(output[key])
    raise TypeError(f"Unsupported MONAI output type: {type(output).__name__}")


class EfficientNetEncoder(nn.Module):
    """Frozen ImageNet EfficientNet-B0 used only as a feature function.

    The 1,000-class ImageNet classifier is replaced by ``Identity``, exposing a
    1,280-dimensional descriptor.  The network is not fine-tuned on the 30
    patient labels, which reduces the risk of severe high-capacity overfitting.
    """

    def __init__(self) -> None:
        """Instantiate the explicit IMAGENET1K_V1 checkpoint and remove its head."""
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        network = models.efficientnet_b0(weights=weights)
        network.classifier = nn.Identity()
        self.network = network.eval()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Return one frozen 1,280-dimensional descriptor per normalized image."""
        return self.network(images)


def efficientnet_normalize(images: torch.Tensor, output_size: int) -> torch.Tensor:
    """Convert aligned grayscale canvases to EfficientNet input tensors.

    One channel is repeated to RGB, the complete square canvas is resized to the
    encoder resolution, and the ImageNet mean/std associated with the pinned
    checkpoint are applied.  Resizing the whole canvas preserves alignment with
    the support mask rather than applying an undocumented center crop.
    """

    if images.shape[1] == 1:
        images = images.repeat(1, 3, 1, 1)
    images = F.interpolate(
        images,
        size=(output_size, output_size),
        mode="bilinear",
        align_corners=False,
    )
    mean = torch.tensor((0.485, 0.456, 0.406), device=images.device).view(1, 3, 1, 1)
    std = torch.tensor((0.229, 0.224, 0.225), device=images.device).view(1, 3, 1, 1)
    return (images - mean) / std


# =============================================================================
# 6. EXACT A17 SUPPORT AND MATCHED CONTROLS
# =============================================================================


def fixed_central_square_mask(
    batch: int,
    height: int,
    width: int,
    fraction: float,
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    """Create the deterministic fallback support used when MONAI is implausible.

    The square size is a fixed fraction of the canvas and is independent of
    label, fold, model score, or predicted mask area.  It prevents an invalid
    segmentation from exposing the entire image while retaining a reproducible
    central field of view.
    """
    side = max(2, min(height, width, int(round(min(height, width) * fraction))))
    top = (height - side) // 2
    left = (width - side) // 2
    mask = torch.zeros(batch, 1, height, width, device=device, dtype=dtype)
    mask[:, :, top : top + side, left : left + side] = 1.0
    return mask


def infer_a17_support(
    monai_model: nn.Module,
    monai_images: torch.Tensor,
    content_mask: torch.Tensor,
    config: Config,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Infer the exact binary support shared by candidate and matched controls.

    The function sums the three non-background MONAI channels, applies fixed
    area/peak plausibility gates, performs the two declared dilations, and uses a
    fixed central fallback on invalid slices.  The final support is intersected
    with the content mask so pipeline-added canvas padding is never visible.

    Returning ``support``, ``valid``, and ``heart_probability`` separately makes
    the scientific contract auditable without recomputing segmentation.
    """

    logits = normalize_monai_output(monai_model(monai_images))
    probabilities = torch.softmax(logits.float(), dim=1)
    heart_probability = probabilities[:, 1:4].sum(dim=1, keepdim=True)

    initial_hard = heart_probability >= config.monai_hard_threshold
    area_ratio = initial_hard.float().mean(dim=(1, 2, 3))
    peak = heart_probability.amax(dim=(1, 2, 3))
    valid = (
        (area_ratio >= config.monai_min_area_ratio)
        & (area_ratio <= config.monai_max_area_ratio)
        & (peak >= config.monai_min_peak_probability)
    )

    dilated = F.max_pool2d(
        initial_hard.float(),
        kernel_size=config.monai_dilation_kernel,
        stride=1,
        padding=config.monai_dilation_kernel // 2,
    )
    dilated = F.max_pool2d(
        dilated,
        kernel_size=config.a17_extra_dilation_kernel,
        stride=1,
        padding=config.a17_extra_dilation_kernel // 2,
    )

    fallback = fixed_central_square_mask(
        batch=monai_images.shape[0],
        height=monai_images.shape[-2],
        width=monai_images.shape[-1],
        fraction=config.fallback_square_fraction,
        device=monai_images.device,
        dtype=monai_images.dtype,
    )
    support = torch.where(valid[:, None, None, None], dilated, fallback)
    support = (support > 0.5).to(monai_images.dtype)
    support = support * (content_mask > 0.5).to(monai_images.dtype)
    return support, valid, heart_probability


def robust_scale_visible_region(
    images: torch.Tensor,
    masks: torch.Tensor,
    config: Config,
) -> torch.Tensor:
    """Robustly normalize each image using only its final visible pixels.

    A batched fixed-bin histogram estimates the declared lower and upper
    percentiles.  The transform excludes hidden pixels and canvas padding, then
    zeroes everything outside the mask.  Consequently, cardiac pixels cannot
    determine the scale of the outside-support control and outside pixels cannot
    determine the scale of the cardiac candidate.

    The minimum visible-pixel and dynamic-range safeguards prevent unstable
    amplification of nearly empty or nearly constant regions.
    """

    if images.ndim != 4 or images.shape[1] != 1:
        raise ValueError("images must have shape [B,1,H,W]")
    if masks.shape != images.shape:
        raise ValueError("masks must align exactly with images")

    batch = images.shape[0]
    bins = config.histogram_bins
    visible = masks > 0.5
    flat_mask = visible[:, 0].reshape(batch, -1)
    counts = flat_mask.sum(dim=1)
    values = images[:, 0].float().clamp(0.0, 1.0)
    indices = torch.round(values * (bins - 1)).long().reshape(batch, -1)

    histogram = torch.zeros(batch, bins, device=images.device, dtype=torch.float32)
    histogram.scatter_add_(1, indices, flat_mask.float())
    cumulative = histogram.cumsum(dim=1)

    safe_counts = counts.clamp_min(1)
    lower_rank = (
        torch.floor((config.lower_percentile / 100.0) * (safe_counts - 1).float()).long()
        + 1
    )
    upper_rank = (
        torch.floor((config.upper_percentile / 100.0) * (safe_counts - 1).float()).long()
        + 1
    )
    lower_bin = (cumulative >= lower_rank[:, None]).long().argmax(dim=1)
    upper_bin = (cumulative >= upper_rank[:, None]).long().argmax(dim=1)
    lower = lower_bin.float() / float(bins - 1)
    upper = upper_bin.float() / float(bins - 1)

    valid = (
        (counts >= config.minimum_visible_pixels)
        & torch.isfinite(lower)
        & torch.isfinite(upper)
        & (upper > lower)
    )
    denominator = (upper - lower).clamp_min(config.minimum_dynamic_range)
    scaled = (images.float() - lower[:, None, None, None]) / denominator[:, None, None, None]
    scaled = scaled.clamp(0.0, 1.0) * visible.float()
    scaled = scaled * valid[:, None, None, None].float()
    return scaled.to(images.dtype)


def coprime_affine_parameters(pixel_hash: str, n: int) -> tuple[int, int]:
    """Derive a deterministic bijection over visible pixel ranks.

    Source rank is ``(a*k+b) mod n``.  Requiring ``gcd(a,n)=1`` guarantees a full
    permutation, so every visible intensity is used exactly once.  Parameters
    depend only on the decoded-pixel hash, never on label, patient, fold, or
    classifier output.
    """

    if n <= 1:
        return 1, 0
    candidate = 2 + (int(pixel_hash[:16], 16) % max(1, n - 2))
    candidate %= n
    candidate = max(candidate, 1)
    start = candidate
    while math.gcd(candidate, n) != 1:
        candidate = 1 if candidate + 1 >= n else candidate + 1
        if candidate == start:
            raise RuntimeError("Could not find a coprime affine multiplier.")
    offset = int(pixel_hash[16:32], 16) % n
    if candidate == 1 and offset == 0:
        offset = 1
    return candidate, offset


def shuffle_inside_support(
    cardiac_images: torch.Tensor,
    support: torch.Tensor,
    pixel_hashes: Sequence[str],
) -> torch.Tensor:
    """Destroy spatial arrangement while preserving support and intensity multiset.

    For every slice, the normalized cardiac values are permuted bijectively
    among the exact support coordinates.  The support boundary, number of
    visible pixels, histogram, mean, variance, quantiles, and extrema remain
    unchanged; only correspondence between intensity and anatomical location is
    destroyed.  Pixels outside the support remain zero.
    """

    output = torch.zeros_like(cardiac_images)
    for index, pixel_hash in enumerate(pixel_hashes):
        positions = torch.nonzero(support[index, 0].reshape(-1) > 0.5).reshape(-1)
        n_values = int(positions.numel())
        if n_values == 0:
            continue
        source = cardiac_images[index, 0].reshape(-1)[positions]
        multiplier, offset = coprime_affine_parameters(pixel_hash, n_values)
        destination = torch.arange(n_values, device=images_device(cardiac_images))
        source_rank = (destination * multiplier + offset) % n_values
        shuffled = source[source_rank]
        for channel in range(3):
            output[index, channel].reshape(-1)[positions] = shuffled
    return output


def images_device(images: torch.Tensor) -> torch.device:
    """Return the tensor device for readable device-local index creation."""

    return images.device


def make_four_views(
    raw_images: torch.Tensor,
    support: torch.Tensor,
    content_mask: torch.Tensor,
    pixel_hashes: Sequence[str],
    config: Config,
) -> dict[str, torch.Tensor]:
    """Construct the candidate and three controls from one exact support.

    ``cardiac`` and ``outside_support`` are independently region-normalized.
    ``support_only`` removes MRI intensity while preserving exact geometry.
    ``shuffled_intensity`` preserves exact geometry and intensity distribution
    while destroying spatial arrangement.  Reusing the same support prevents
    candidate/control drift.
    """

    cardiac_gray = robust_scale_visible_region(raw_images, support, config)
    outside_mask = ((content_mask > 0.5) & (support <= 0.5)).float()
    outside_gray = robust_scale_visible_region(raw_images, outside_mask, config)

    cardiac_rgb = cardiac_gray.repeat(1, 3, 1, 1)
    support_rgb = support.repeat(1, 3, 1, 1)
    shuffled_rgb = shuffle_inside_support(cardiac_rgb, support, pixel_hashes)
    outside_rgb = outside_gray.repeat(1, 3, 1, 1)
    return {
        "cardiac": cardiac_rgb,
        "support_only": support_rgb,
        "shuffled_intensity": shuffled_rgb,
        "outside_support": outside_rgb,
    }


# =============================================================================
# 7. COMPACT SERIES-LEVEL FEATURE BANK
# =============================================================================


def extract_series_bank(config: Config, records: Sequence[SliceRecord]) -> SeriesBank:
    """Run frozen segmentation/encoding once and accumulate exact series means.

    On a cache miss this is the expensive stage.  It:
    * creates a deterministic DataLoader;
    * loads verified MONAI and frozen EfficientNet models;
    * builds the four matched views per batch;
    * encodes all views in one concatenated call;
    * accumulates float64 sums by series and stores float32 means;
    * accumulates metadata and slice counts;
    * computes exact/pHash duplicate audits;
    * writes a compact NPZ cache and JSON manifest.

    Labels are carried only to verify consistent patient/series membership and
    are never supplied to either neural network.
    """

    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = configuration_fingerprint(config, records)
    cache_path = output_dir / f"series_feature_bank_{fingerprint}.npz"
    manifest_path = output_dir / f"series_feature_bank_{fingerprint}.json"

    if cache_path.is_file() and manifest_path.is_file() and not config.force_rebuild_cache:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("schema") != FEATURE_CACHE_SCHEMA or manifest.get("fingerprint") != fingerprint:
            raise RuntimeError("Cache manifest identity mismatch; set MAIN_FORCE_REBUILD_CACHE=True.")
        if "duplicate_audit" not in manifest:
            raise RuntimeError("Cache lacks its duplicate audit; rebuild instead of bypassing the audit.")
        audit = manifest["duplicate_audit"]
        write_json(output_dir / "duplicate_audit.json", audit)
        if audit.get("cross_patient_exact_groups") and config.fail_on_cross_patient_exact_duplicate:
            raise RuntimeError("Cached cross-patient exact duplicates require review before evaluation.")
        cached = load_series_bank(cache_path)
        validate_series_bank_records(cached, records)
        print(f"[CACHE] Loading {cache_path}", flush=True)
        return cached

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[MODEL] device={device}", flush=True)
    monai_model = load_monai_segmenter(config, device)
    encoder = EfficientNetEncoder().to(device).eval()

    dataset = SliceDataset(records, config)
    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=(device.type == "cuda"),
    )

    feature_sums: dict[str, dict[str, np.ndarray]] = {
        mode: {} for mode in FEATURE_MODES
    }
    metadata_sums: dict[str, np.ndarray] = {}
    series_counts: dict[str, int] = defaultdict(int)
    series_patient: dict[str, str] = {}
    series_label: dict[str, int] = {}

    hash_to_patients: dict[str, set[str]] = defaultdict(set)
    phash_to_patients: dict[int, set[str]] = defaultdict(set)

    autocast_enabled = bool(config.use_amp and device.type == "cuda")
    started = time.perf_counter()
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Extracting four views", unit="batch"):
            raw = batch["raw"].to(device, non_blocking=True)
            monai = batch["monai"].to(device, non_blocking=True)
            content = batch["content"].to(device, non_blocking=True)
            pixel_hashes = list(batch["pixel_hash"])

            support, valid, _ = infer_a17_support(
                monai_model, monai, content, config
            )
            views = make_four_views(raw, support, content, pixel_hashes, config)
            normalized = [
                efficientnet_normalize(views[mode], config.encoder_input_size)
                for mode in FEATURE_MODES
            ]
            stacked = torch.cat(normalized, dim=0)
            with torch.autocast(
                device_type=device.type,
                enabled=autocast_enabled,
                dtype=torch.float16 if device.type == "cuda" else torch.float32,
            ):
                embeddings = encoder(stacked).float()
            split_embeddings = embeddings.split(raw.shape[0], dim=0)

            support_fraction = support.mean(dim=(1, 2, 3)).cpu().numpy()
            valid_np = valid.float().cpu().numpy()
            base_metadata = batch["metadata"].cpu().numpy().astype(np.float64)
            complete_metadata = np.column_stack(
                [base_metadata, valid_np, support_fraction]
            )

            labels = batch["label"].cpu().numpy().astype(np.int64)
            patient_ids = list(batch["patient_id"])
            series_ids = list(batch["series_id"])
            phashes = [int(value, 16) for value in batch["phash"]]

            for row_index, (patient_id, series_id, label) in enumerate(
                zip(patient_ids, series_ids, labels)
            ):
                patient_id = str(patient_id)
                series_id = str(series_id)
                label = int(label)
                previous_patient = series_patient.get(series_id)
                previous_label = series_label.get(series_id)
                if previous_patient is not None and previous_patient != patient_id:
                    raise RuntimeError(f"Series ID collision: {series_id}")
                if previous_label is not None and previous_label != label:
                    raise RuntimeError(f"Series label conflict: {series_id}")
                series_patient[series_id] = patient_id
                series_label[series_id] = label
                series_counts[series_id] += 1

                if series_id not in metadata_sums:
                    metadata_sums[series_id] = np.zeros(
                        len(METADATA_NAMES), dtype=np.float64
                    )
                metadata_sums[series_id] += complete_metadata[row_index]

                for mode, tensor in zip(FEATURE_MODES, split_embeddings):
                    vector = tensor[row_index].detach().cpu().numpy().astype(np.float64)
                    if series_id not in feature_sums[mode]:
                        feature_sums[mode][series_id] = np.zeros(
                            config.embedding_dim, dtype=np.float64
                        )
                    feature_sums[mode][series_id] += vector

                hash_to_patients[pixel_hashes[row_index]].add(patient_id)
                phash_to_patients[int(phashes[row_index])].add(patient_id)

    series_ids = np.asarray(sorted(series_counts))
    patient_ids = np.asarray([series_patient[sid] for sid in series_ids])
    labels = np.asarray([series_label[sid] for sid in series_ids], dtype=np.int64)
    counts = np.asarray([series_counts[sid] for sid in series_ids], dtype=np.int64)
    metadata = np.stack(
        [metadata_sums[sid] / series_counts[sid] for sid in series_ids]
    ).astype(np.float32)
    features = {
        mode: np.stack(
            [feature_sums[mode][sid] / series_counts[sid] for sid in series_ids]
        ).astype(np.float32)
        for mode in FEATURE_MODES
    }

    cross_patient_exact = {
        digest: sorted(patients)
        for digest, patients in hash_to_patients.items()
        if len(patients) > 1
    }

    audit = {
        "decoded_slices": len(records),
        "series_rows": int(len(series_ids)),
        "exact_hash_groups": int(len(hash_to_patients)),
        "cross_patient_exact_groups": cross_patient_exact,
        "perceptual_hash_unique_values": int(len(phash_to_patients)),
    }
    if config.run_perceptual_hash_audit:
        patient_label_map = {
            patient: int(label)
            for patient, label in zip(patient_ids.tolist(), labels.tolist())
        }
        candidates = phash_patient_pairs(
            phash_to_patients,
            config.phash_hamming_radius,
            patient_label_map,
        )
        audit["perceptual_patient_pair_candidates"] = candidates
        audit["perceptual_candidate_pair_count"] = len(candidates)
        audit["perceptual_cross_label_pair_count"] = sum(
            int(row["cross_label"]) for row in candidates
        )
    write_json(output_dir / "duplicate_audit.json", audit)
    if cross_patient_exact and config.fail_on_cross_patient_exact_duplicate:
        raise RuntimeError(
            "Confirmed decoded-pixel duplicates cross patients. "
            "Inspect duplicate_audit.json before evaluation."
        )

    np.savez(
        cache_path,
        series_ids=series_ids,
        patient_ids=patient_ids,
        labels=labels,
        n_slices=counts,
        metadata=metadata,
        **{f"features_{mode}": features[mode] for mode in FEATURE_MODES},
    )
    write_json(
        manifest_path,
        {
            "fingerprint": fingerprint,
            "schema": FEATURE_CACHE_SCHEMA,
            "duplicate_audit": audit,
            "config": asdict(config),
            "feature_modes": list(FEATURE_MODES),
            "metadata_names": list(METADATA_NAMES),
            "elapsed_seconds": time.perf_counter() - started,
            "cache_path": str(cache_path),
        },
    )
    print(
        f"[CACHE] Wrote {len(series_ids)} series rows in "
        f"{time.perf_counter() - started:.1f}s",
        flush=True,
    )
    bank = SeriesBank(series_ids, patient_ids, labels, counts, features, metadata)
    validate_series_bank_records(bank, records)
    return bank


def validate_series_bank_records(bank: SeriesBank, records: Sequence[SliceRecord]) -> None:
    """Reject cache rows whose grouping/count/label contract differs from discovery."""
    expected: dict[str, list[Any]] = {}
    for row in records:
        if row.series_id not in expected:
            expected[row.series_id] = [row.patient_id, row.label, 0]
        entry = expected[row.series_id]
        if entry[:2] != [row.patient_id, row.label]:
            raise RuntimeError("Conflicting series ownership.")
        entry[2] += 1
    if len(set(bank.series_ids.tolist())) != len(bank.series_ids) or set(expected) != set(bank.series_ids.tolist()):
        raise RuntimeError("Series cache membership does not match the discovered manifest.")
    for sid, pid, label, n in zip(bank.series_ids, bank.patient_ids, bank.labels, bank.n_slices):
        if expected[str(sid)] != [str(pid), int(label), int(n)]:
            raise RuntimeError(f"Series cache contract mismatch: {sid}")
    if len(bank.metadata) != len(expected) or not np.isfinite(bank.metadata).all():
        raise RuntimeError("Invalid cached metadata rows.")
    for mode in FEATURE_MODES:
        if len(bank.features[mode]) != len(expected) or not np.isfinite(bank.features[mode]).all():
            raise RuntimeError(f"Invalid cached features: {mode}")


def load_series_bank(path: Path) -> SeriesBank:
    """Reconstruct a ``SeriesBank`` from the compact NPZ cache.

    The loader restores fixed dtypes and all four declared modes.  The JSON
    manifest and configuration fingerprint are checked by the caller before this
    function is used.
    """
    data = np.load(path, allow_pickle=False)
    return SeriesBank(
        series_ids=data["series_ids"],
        patient_ids=data["patient_ids"],
        labels=data["labels"].astype(np.int64),
        n_slices=data["n_slices"].astype(np.int64),
        features={mode: data[f"features_{mode}"].astype(np.float32) for mode in FEATURE_MODES},
        metadata=data["metadata"].astype(np.float32),
    )


class BKTree:
    """Minimal Burkhard-Keller tree for exact Hamming-radius search.

    Hamming distance over 64-bit pHashes is a metric, so BK-tree triangle-inequality
    bounds can prune most comparisons without truncating the search.  The tree
    returns screening candidates only; it does not declare clinical images to be
    duplicates.
    """

    def __init__(self) -> None:
        """Initialize an empty metric tree."""
        self.root: tuple[int, dict[int, Any]] | None = None

    @staticmethod
    def distance(left: int, right: int) -> int:
        """Return exact 64-bit Hamming distance using XOR bit count."""
        return int((left ^ right).bit_count())

    def add(self, value: int) -> None:
        """Insert one unique hash into the metric tree without rebalancing."""
        if self.root is None:
            self.root = (value, {})
            return
        node = self.root
        while True:
            current, children = node
            distance = self.distance(value, current)
            if distance == 0:
                return
            if distance not in children:
                children[distance] = (value, {})
                return
            node = children[distance]

    def query(self, value: int, radius: int) -> list[int]:
        """Return every stored hash within the inclusive radius.

        Child edges outside ``distance ± radius`` cannot contain valid matches by
        the triangle inequality and are safely pruned.
        """
        if self.root is None:
            return []
        matches: list[int] = []
        stack = [self.root]
        while stack:
            current, children = stack.pop()
            distance = self.distance(value, current)
            if distance <= radius:
                matches.append(current)
            lower, upper = distance - radius, distance + radius
            stack.extend(
                child
                for edge, child in children.items()
                if lower <= edge <= upper
            )
        return matches


def phash_patient_pairs(
    phash_to_patients: dict[int, set[str]],
    radius: int,
    patient_to_label: dict[str, int],
) -> list[dict[str, Any]]:
    """Convert near-hash matches into deduplicated patient-pair candidates.

    For each patient pair, the minimum observed pHash distance is retained and a
    cross-label flag is recorded.  The result is an audit queue for manual review,
    not evidence that two patients or images are identical.
    """

    tree = BKTree()
    values = sorted(phash_to_patients)
    for value in values:
        tree.add(value)
    pairs: dict[tuple[str, str], int] = {}
    for value in values:
        for neighbor in tree.query(value, radius):
            if neighbor < value:
                continue
            distance = BKTree.distance(value, neighbor)
            for left in phash_to_patients[value]:
                for right in phash_to_patients[neighbor]:
                    if left == right:
                        continue
                    pair = tuple(sorted((left, right)))
                    pairs[pair] = min(distance, pairs.get(pair, 65))
    return [
        {
            "patient_a": left,
            "patient_b": right,
            "minimum_hamming_distance": distance,
            "cross_label": bool(
                patient_to_label[left] != patient_to_label[right]
            ),
        }
        for (left, right), distance in sorted(pairs.items())
    ]


# =============================================================================
# 8. SERIES -> PATIENT POOLING AND NUISANCE MATRIX
# =============================================================================


def apply_reviewed_series_annotations(
    series: SeriesBank,
    config: Config,
) -> tuple[SeriesBank, dict[str, Any]]:
    """Optionally retain reviewed cardiac sequence/view cells common to ALL patients.

    The blank template is always written. The input CSV must contain series_id,
    sequence_type, view_type, contains_heart, is_localizer, is_derived_export.
    Boolean fields require explicit true/false (or 1/0). Blank/incomplete rows are
    excluded, never guessed. Unknown or duplicate IDs raise an error because an
    annotation made under immediate_parent grouping cannot be silently reused
    under patient_child. No patient is removed to manufacture a common protocol.

    Selection uses anatomy/protocol annotations, not class labels. The strict
    common-cell intersection is a label-blind cohort restriction, still not an
    external test. Different scan settings may remain within a nominal cell.
    """
    out = Path(config.output_dir)
    columns = ["series_id", "patient_id", "n_slices", "sequence_type", "view_type",
               "contains_heart", "is_localizer", "is_derived_export"]
    with (out / "series_annotation_template.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        for sid, pid, n in zip(series.series_ids, series.patient_ids, series.n_slices):
            writer.writerow([str(sid), str(pid), int(n), "", "", "", "", ""])
    if not config.series_annotations_path:
        status = {"status": "SKIPPED_NO_ANNOTATIONS", "reason": "No manually reviewed sequence/view CSV supplied."}
        write_json(out / "series_annotation_status.json", status)
        return series, status
    known = set(map(str, series.series_ids))
    by_id: dict[str, str] = {}
    seen: set[str] = set()
    def boolean(value: str) -> bool | None:
        """Parse only explicit annotation booleans; blanks mean unreviewed."""
        token = str(value).strip().lower()
        if token in {"true", "1", "yes"}:
            return True
        if token in {"false", "0", "no"}:
            return False
        if not token:
            return None
        raise ValueError(f"Invalid annotation boolean: {value!r}")
    with Path(config.series_annotations_path).open(newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        required = set(columns) - {"patient_id", "n_slices"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"Annotation CSV requires columns: {sorted(required)}")
        for row in reader:
            sid = str(row["series_id"]).strip()
            if sid not in known or sid in seen:
                raise ValueError(f"Unknown/duplicate annotation series_id {sid!r}; regenerate the template for this grouping.")
            seen.add(sid)
            heart, localizer, derived = [boolean(row[k]) for k in
                                        ("contains_heart", "is_localizer", "is_derived_export")]
            seq, view = [str(row[k]).strip().lower() for k in ("sequence_type", "view_type")]
            if heart is True and localizer is False and derived is False and seq and view:
                by_id[sid] = f"{seq}::{view}"
    patients = sorted(set(series.patient_ids.tolist()))
    cells_by_patient = {pid: {by_id[str(series.series_ids[i])] for i in
                       np.flatnonzero(series.patient_ids == pid) if str(series.series_ids[i]) in by_id}
                       for pid in patients}
    common = set.intersection(*(cells_by_patient[p] for p in patients))
    if not common:
        status = {"status": "FAILED_NO_COMMON_REVIEWED_CELLS", "cells_by_patient":
                  {str(k): sorted(v) for k, v in cells_by_patient.items()},
                  "reason": "No reviewed sequence/view cell exists for every patient. No automatic patient dropping is permitted."}
        write_json(out / "series_annotation_status.json", status)
        raise ValueError(status["reason"])
    keep = np.asarray([by_id.get(str(sid)) in common for sid in series.series_ids], dtype=bool)
    cells = np.asarray([by_id[str(sid)] for sid in series.series_ids[keep]])
    filtered = SeriesBank(series.series_ids[keep], series.patient_ids[keep], series.labels[keep],
                          series.n_slices[keep], {k: v[keep] for k, v in series.features.items()},
                          series.metadata[keep], cells)
    if set(filtered.patient_ids.tolist()) != set(patients):
        raise RuntimeError("Annotation filter unexpectedly changed the patient cohort.")
    status = {"status": "APPLIED_COMMON_CELLS", "input_sha256": sha256_file(Path(config.series_annotations_path)),
              "common_sequence_view_cells": sorted(common), "patients_retained": len(patients),
              "series_before": len(series.series_ids), "series_retained": int(keep.sum()),
              "slices_retained": int(filtered.n_slices.sum()), "pooling": "slice -> series -> equal cell -> patient"}
    write_json(out / "series_annotation_status.json", status)
    return filtered, status


def nuisance_block(bank: PatientBank, scope: str) -> np.ndarray:
    """Return explicitly named nuisance variables without learning from the cohort.

    export = geometry/bytes/padding/count metadata; anatomy_proxy = projected
    support/outside plus MONAI gate/support area; combined = both. Neither group
    is a certified causal confounder set. A missing block is NOT silently replaced
    by combined because that would change the scientific experiment.
    """
    if bank.nuisance_blocks is None:
        if scope == "combined":
            return bank.nuisance
        raise ValueError("Separated nuisance blocks require build_patient_bank().")
    return bank.nuisance_blocks[scope]


def build_patient_bank(
    series: SeriesBank, nuisance_projection_dim: int, nuisance_projection_seed: int
) -> PatientBank:
    """Pool series equally to patients and build the nuisance design matrix.

    Each patient feature is the unweighted mean of its series-proxy means.  The
    nuisance matrix contains fixed random projections of exact support-only and
    outside-support embeddings plus simple acquisition/export metadata and series
    statistics.  The random projection is label-independent and fixed before
    evaluation; fold-local scaling and PCA are still fitted later.

    This design makes the confounding hypothesis explicit while keeping the
    nuisance dimension appropriate for only 24 outer-training patients.
    """

    patient_ids = np.asarray(sorted(set(series.patient_ids.tolist())))
    patient_to_rows = {
        patient_id: np.flatnonzero(series.patient_ids == patient_id)
        for patient_id in patient_ids
    }
    labels = []
    patient_features = {mode: [] for mode in FEATURE_MODES}
    patient_metadata = []

    for patient_id in patient_ids:
        rows = patient_to_rows[patient_id]
        patient_labels = np.unique(series.labels[rows])
        if len(patient_labels) != 1:
            raise RuntimeError(f"Inconsistent labels for {patient_id}")
        labels.append(int(patient_labels[0]))
        for mode in FEATURE_MODES:
            if series.pooling_cells is None:
                patient_features[mode].append(series.features[mode][rows].mean(axis=0))
            else:
                # Equal sequence/view-cell mass; equal series weights inside each cell.
                cells = series.pooling_cells[rows]
                cell_means = [series.features[mode][rows[cells == cell]].mean(axis=0)
                              for cell in sorted(set(cells.tolist()))]
                patient_features[mode].append(np.mean(cell_means, axis=0))
        metadata_mean = series.metadata[rows].mean(axis=0)
        metadata_augmented = np.concatenate(
            [
                metadata_mean,
                np.asarray(
                    [
                        float(len(rows)),
                        float(series.n_slices[rows].sum()),
                        float(series.n_slices[rows].mean()),
                        float(series.n_slices[rows].max()),
                    ],
                    dtype=np.float32,
                ),
            ]
        )
        patient_metadata.append(metadata_augmented)

    feature_arrays = {
        mode: np.asarray(values, dtype=np.float32)
        for mode, values in patient_features.items()
    }
    metadata_array = np.asarray(patient_metadata, dtype=np.float32)
    # A fixed Johnson-Lindenstrauss-style random projection keeps the nuisance
    # matrix compact without learning from labels or from the cohort.  Fold-local
    # scaling/PCA is still fitted later.  This is far faster than repeatedly
    # decomposing 2,560 control dimensions inside every inner fold.
    projection_dim = max(4, min(64, int(nuisance_projection_dim)))
    rng = np.random.default_rng(nuisance_projection_seed)
    scale = 1.0 / math.sqrt(projection_dim)
    support_projection = rng.normal(
        0.0, scale, size=(feature_arrays["support_only"].shape[1], projection_dim)
    ).astype(np.float32)
    outside_projection = rng.normal(
        0.0, scale, size=(feature_arrays["outside_support"].shape[1], projection_dim)
    ).astype(np.float32)
    nuisance = np.concatenate(
        [
            feature_arrays["support_only"] @ support_projection,
            feature_arrays["outside_support"] @ outside_projection,
            metadata_array,
        ],
        axis=1,
    )
    return PatientBank(
        patient_ids=patient_ids,
        labels=np.asarray(labels, dtype=np.int64),
        features=feature_arrays,
        nuisance=nuisance.astype(np.float32),
        metadata=metadata_array,
        patient_to_series_rows=patient_to_rows,
        nuisance_blocks={
            "export": metadata_array[:, [0, 1, 2, 3, 4, 5, 8, 9, 10, 11]].astype(np.float32),
            "anatomy_proxy": np.concatenate([
                feature_arrays["support_only"] @ support_projection,
                feature_arrays["outside_support"] @ outside_projection,
                metadata_array[:, [6, 7]],
            ], axis=1).astype(np.float32),
            "combined": nuisance.astype(np.float32),
        },
    )


# =============================================================================
# 9. FOLD-LOCAL LINEAR ALGEBRA
# =============================================================================


class CompactPCA:
    """Fold-local standardization followed by a deliberately small PCA.

    The retained dimension is the smaller of the declared variance requirement,
    maximum component cap, feature count, and training-sample constraint.  Both
    scaler and PCA are fitted only on the current training partition.
    """

    def __init__(self, variance: float, maximum_components: int):
        """Store the variance target and maximum component cap."""
        self.variance = variance
        self.maximum_components = maximum_components
        self.scaler = StandardScaler()
        self.pca: PCA | None = None

    def fit(self, X: np.ndarray) -> "CompactPCA":
        """Fit StandardScaler, probe explained variance, and refit the chosen PCA."""
        Xs = self.scaler.fit_transform(X)
        maximum = max(1, min(self.maximum_components, Xs.shape[0] - 2, Xs.shape[1]))
        probe = PCA(n_components=maximum, svd_solver="full", random_state=0).fit(Xs)
        cumulative = np.cumsum(probe.explained_variance_ratio_)
        required = int(np.searchsorted(cumulative, self.variance) + 1)
        components = max(1, min(maximum, required))
        self.pca = PCA(n_components=components, svd_solver="full", random_state=0).fit(Xs)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply the already-fitted scaler and PCA without refitting."""
        if self.pca is None:
            raise RuntimeError("CompactPCA must be fit before transform.")
        return self.pca.transform(self.scaler.transform(X))

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Fit on the supplied training matrix and return its compact coordinates."""
        return self.fit(X).transform(X)


class NuisanceEncoder:
    """Fold-local compression of nuisance controls into a few linear axes.

    Keeping the nuisance subspace small stabilizes residualization and the
    minimax penalty when an outer fold contains only about 24 training patients.
    """

    def __init__(self, maximum_components: int):
        """Store the maximum nuisance-axis count and create an unfitted scaler."""
        self.maximum_components = maximum_components
        self.scaler = StandardScaler()
        self.pca: PCA | None = None

    def fit(self, Z: np.ndarray) -> "NuisanceEncoder":
        """Fit nuisance scaling and full-SVD PCA on training patients only."""
        Zs = self.scaler.fit_transform(Z)
        components = max(1, min(self.maximum_components, Zs.shape[0] - 2, Zs.shape[1]))
        self.pca = PCA(n_components=components, svd_solver="full", random_state=0).fit(Zs)
        return self

    def transform(self, Z: np.ndarray) -> np.ndarray:
        """Project nuisance variables through the fitted training-only transform."""
        if self.pca is None:
            raise RuntimeError("NuisanceEncoder must be fit before transform.")
        return self.pca.transform(self.scaler.transform(Z))


class NuisanceResidualizer:
    """Remove cardiac variation that is linearly reconstructable from nuisance.

    For centered cardiac features ``Xc`` and compressed nuisance features ``Zc``,
    solve

        B* = argmin_B ||Xc - Zc B||_F^2 + alpha ||B||_F^2

    and return ``Xc - rho Zc B*`` with an explicit fraction rho in [0,1].  Ridge regularization prevents the unstable exact
    projection that would arise from a tiny training cohort and many controls.
    Every mean, scaler, PCA axis, and coefficient is learned inside the current
    training fold.  The operation is a sensitivity analysis, not a proof of
    causal deconfounding.
    """

    def __init__(self, alpha: float, max_nuisance_components: int, rho: float = 1.0):
        """Create an unfitted ridge residualizer and fold-local nuisance encoder."""
        self.alpha = float(alpha)
        self.rho = float(rho)
        if not 0.0 <= self.rho <= 1.0:
            raise ValueError("rho must lie in [0, 1].")
        self.encoder = NuisanceEncoder(max_nuisance_components)
        self.x_mean: np.ndarray | None = None
        self.coefficients: np.ndarray | None = None

    def fit(self, X: np.ndarray, Z: np.ndarray) -> "NuisanceResidualizer":
        """Fit nuisance axes and solve the regularized normal equations."""
        self.x_mean = np.asarray(X, dtype=np.float64).mean(axis=0)
        Zc = self.encoder.fit(Z).transform(Z).astype(np.float64)
        Xc = np.asarray(X, dtype=np.float64) - self.x_mean
        gram = Zc.T @ Zc + self.alpha * np.eye(Zc.shape[1])
        self.coefficients = np.linalg.solve(gram, Zc.T @ Xc)
        return self

    def transform(self, X: np.ndarray, Z: np.ndarray) -> np.ndarray:
        """Subtract rho times the training-estimated nuisance component from new rows."""
        if self.x_mean is None or self.coefficients is None:
            raise RuntimeError("NuisanceResidualizer is not fitted.")
        Zc = self.encoder.transform(Z).astype(np.float64)
        return (
            np.asarray(X, dtype=np.float64)
            - self.x_mean
            - self.rho * (Zc @ self.coefficients)
        ).astype(np.float32)


# =============================================================================
# 10. CUSTOM MINIMAX-INSPIRED CONFOUNDER-GUARDED LOGISTIC REGRESSION
# =============================================================================


class WorstCaseLinearConfounderLogistic:
    """Logistic regression with a closed-form worst linear nuisance adversary.

    With score ``s=Xw+b`` and centered nuisance axes ``Z``, the strongest squared
    covariance over all unit nuisance combinations equals

        || Z^T (s-mean(s)) / n ||_2^2.

    The optimized objective is balanced logistic loss plus this covariance
    penalty and L2 regularization.  Explicit gradients and backtracking line
    search keep the implementation interpretable and suitable for a very small
    patient cohort.  Only modeled linear score dependence is controlled.
    """

    def __init__(
        self,
        confounder_lambda: float,
        l2: float,
        learning_rate: float,
        max_iterations: int,
        tolerance: float,
    ) -> None:
        """Store optimization constants and initialize an unfitted linear model."""
        self.confounder_lambda = float(confounder_lambda)
        self.l2 = float(l2)
        self.learning_rate = float(learning_rate)
        self.max_iterations = int(max_iterations)
        self.tolerance = float(tolerance)
        self.weights: np.ndarray | None = None
        self.bias: float = 0.0
        self.training_trace: list[float] = []

    def _loss_and_gradient(
        self,
        X: np.ndarray,
        Z: np.ndarray,
        y: np.ndarray,
        weights: np.ndarray,
        bias: float,
    ) -> tuple[float, np.ndarray, float]:
        """Evaluate the full objective and its analytical gradients.

        Class-balanced weights prevent the 16/14 class split from changing the
        objective simply through frequency.  The nuisance term uses the closed-form
        maximizing covariance vector, so no separately trained adversarial network
        is required.
        """
        n = len(y)
        scores = X @ weights + bias
        probabilities = stable_sigmoid(scores)

        # Balanced observation weights keep the two patient classes comparable.
        positives = max(1, int(np.sum(y == 1)))
        negatives = max(1, int(np.sum(y == 0)))
        sample_weight = np.where(y == 1, n / (2.0 * positives), n / (2.0 * negatives))
        normalizer = float(sample_weight.sum())
        logistic_terms = np.logaddexp(0.0, scores) - y * scores
        logistic_loss = float(np.dot(sample_weight, logistic_terms) / normalizer)
        grad_scores = sample_weight * (probabilities - y) / normalizer

        # Z is centered by the fold-local nuisance encoder.  The expression below
        # is the maximizing linear nuisance direction in closed form.
        centered_scores = scores - scores.mean()
        centered_Z = Z - Z.mean(axis=0, keepdims=True)
        covariance_vector = centered_Z.T @ centered_scores / float(n)
        confounder_penalty = self.confounder_lambda * float(
            covariance_vector @ covariance_vector
        )
        grad_scores += (
            2.0
            * self.confounder_lambda
            / float(n)
            * (centered_Z @ covariance_vector)
        )

        regularization = 0.5 * self.l2 * float(weights @ weights)
        loss = logistic_loss + confounder_penalty + regularization
        grad_w = X.T @ grad_scores + self.l2 * weights
        grad_b = float(np.sum(sample_weight * (probabilities - y) / normalizer))
        return loss, grad_w, grad_b

    def fit(
        self,
        X: np.ndarray,
        Z: np.ndarray,
        y: np.ndarray,
    ) -> "WorstCaseLinearConfounderLogistic":
        """Optimize weights with batch gradient descent and Armijo backtracking.

        The model starts at zero, stops on small gradient norm or loss improvement,
        and records every accepted objective value.  Candidate step sizes are reduced
        until they satisfy a sufficient-decrease condition, avoiding a fixed step
        that may diverge for one fold.
        """
        X = np.asarray(X, dtype=np.float64)
        Z = np.asarray(Z, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        weights = np.zeros(X.shape[1], dtype=np.float64)
        bias = 0.0
        learning_rate = self.learning_rate

        loss, grad_w, grad_b = self._loss_and_gradient(X, Z, y, weights, bias)
        self.training_trace = [loss]
        for _ in range(self.max_iterations):
            gradient_norm = math.sqrt(float(grad_w @ grad_w) + grad_b * grad_b)
            if gradient_norm < self.tolerance:
                break

            accepted = False
            trial_rate = learning_rate
            for _ in range(25):
                candidate_w = weights - trial_rate * grad_w
                candidate_b = bias - trial_rate * grad_b
                candidate_loss, candidate_grad_w, candidate_grad_b = self._loss_and_gradient(
                    X, Z, y, candidate_w, candidate_b
                )
                if candidate_loss <= loss - 1e-4 * trial_rate * gradient_norm**2:
                    weights, bias = candidate_w, candidate_b
                    loss, grad_w, grad_b = (
                        candidate_loss,
                        candidate_grad_w,
                        candidate_grad_b,
                    )
                    learning_rate = min(self.learning_rate, trial_rate * 1.05)
                    accepted = True
                    break
                trial_rate *= 0.5
            if not accepted:
                break
            self.training_trace.append(loss)
            if len(self.training_trace) > 2 and abs(
                self.training_trace[-2] - self.training_trace[-1]
            ) < self.tolerance:
                break

        self.weights = weights
        self.bias = float(bias)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return two-column logistic scores after verifying the model is fitted."""
        if self.weights is None:
            raise RuntimeError("The model must be fitted before prediction.")
        probability = stable_sigmoid(np.asarray(X) @ self.weights + self.bias)
        return np.column_stack([1.0 - probability, probability])


# =============================================================================
# 11. RELIABILITY-WEIGHTED LOGIT FUSION (NO PRIOR)
# =============================================================================


def fuse_series_logits(logits: np.ndarray, n_slices: np.ndarray, tau: float) -> float:
    """Return sigmoid(sum(r*logit)/sum(r)), r=n/(n+tau), with no prior claim.

    In the old expression prior_logit + mean(logit-prior_logit), the prior
    cancels algebraically. This method is explicitly a weighted logit mean, NOT
    Bayesian inference and NOT calibrated clinical evidence. Raw decision logits
    avoid clipping probabilities before the inverse logistic transform.

    tau=0 gives equal weights; larger tau increases relative preference for long
    series. Duplicating ALL series equally leaves the result unchanged. Duplicating
    one series can change its relative mass: this is not a deduplication method.
    """
    logits = np.asarray(logits, dtype=np.float64)
    counts = np.asarray(n_slices, dtype=np.float64)
    if (logits.ndim != 1 or counts.shape != logits.shape or not len(logits)
            or not np.isfinite(logits).all() or not np.isfinite(counts).all()
            or np.any(counts <= 0) or not np.isfinite(tau) or tau < 0):
        raise ValueError("Expected finite aligned logits, positive counts, and tau>=0.")
    weights = counts / (counts + float(tau))
    return float(stable_sigmoid(np.asarray([np.average(logits, weights=weights)]))[0])


def reliability_patient_embeddings(
    series: SeriesBank,
    patient_ids: Sequence[str],
    tau: float,
) -> np.ndarray:
    """Use the SAME reliability pooling at training and inference.

    For an annotated run, each common sequence/view cell receives equal mass;
    reliability weights operate only inside a cell. tau=0 then reduces exactly
    to build_patient_bank's baseline pooling, apart from floating-point roundoff.
    """
    if tau < 0 or not np.isfinite(tau):
        raise ValueError("tau must be finite and nonnegative.")
    vectors = []
    for pid in patient_ids:
        rows = np.flatnonzero(series.patient_ids == pid)
        if not len(rows):
            raise ValueError(f"No series for patient {pid}")
        cells = np.zeros(len(rows), dtype=int) if series.pooling_cells is None else series.pooling_cells[rows]
        cell_vectors = []
        for cell in sorted(set(cells.tolist())):
            indices = rows[cells == cell]
            n = series.n_slices[indices].astype(np.float64)
            weights = n / (n + tau)
            cell_vectors.append(np.average(series.features["cardiac"][indices], axis=0, weights=weights))
        vectors.append(np.mean(cell_vectors, axis=0))
    return np.asarray(vectors, dtype=np.float32)


# =============================================================================
# 12. MODEL FIT/PREDICT ROUTINES
# =============================================================================


def fit_standard_logistic(
    X: np.ndarray,
    y: np.ndarray,
    C: float,
    config: Config,
) -> tuple[CompactPCA, LogisticRegression]:
    """Fit the standard compact patient-level linear baseline.

    Standardization and PCA are training-only.  Logistic Regression uses balanced
    class weights and explicit regularization ``C`` selected by inner CV.  The
    returned tuple contains every object required to transform validation rows
    without refitting.
    """
    preprocessor = CompactPCA(config.pca_variance, config.max_pca_components)
    transformed = preprocessor.fit_transform(X)
    model = LogisticRegression(
        C=float(C),
        class_weight="balanced",
        solver="liblinear",
        max_iter=4000,
        random_state=config.seed,
    ).fit(transformed, y)
    return preprocessor, model


def predict_standard_logistic(
    fitted: tuple[CompactPCA, LogisticRegression], X: np.ndarray
) -> np.ndarray:
    """Apply a fitted compact PCA/logistic pipeline and return positive-class scores."""
    preprocessor, model = fitted
    return model.predict_proba(preprocessor.transform(X))[:, 1]


def fit_residualized_logistic(
    X: np.ndarray,
    Z: np.ndarray,
    y: np.ndarray,
    alpha: float,
    C: float,
    config: Config,
    rho: float = 1.0,
) -> tuple[NuisanceResidualizer, CompactPCA, LogisticRegression]:
    """Fit nuisance residualization followed by the standard logistic pipeline.

    Both the residualizer and classifier are fitted on the same training
    partition.  Validation nuisance variables are used only to apply the already
    learned subtraction, never to estimate coefficients.
    """
    residualizer = NuisanceResidualizer(alpha, config.max_nuisance_components, rho=rho).fit(X, Z)
    cleaned = residualizer.transform(X, Z)
    preprocessor, model = fit_standard_logistic(cleaned, y, C, config)
    return residualizer, preprocessor, model


def predict_residualized_logistic(
    fitted: tuple[NuisanceResidualizer, CompactPCA, LogisticRegression],
    X: np.ndarray,
    Z: np.ndarray,
) -> np.ndarray:
    """Apply the training-fitted residualizer and logistic classifier to new rows."""
    residualizer, preprocessor, model = fitted
    cleaned = residualizer.transform(X, Z)
    return model.predict_proba(preprocessor.transform(cleaned))[:, 1]


def fit_guarded_logistic(
    X: np.ndarray,
    Z: np.ndarray,
    y: np.ndarray,
    confounder_lambda: float,
    l2: float,
    config: Config,
) -> tuple[CompactPCA, NuisanceEncoder, WorstCaseLinearConfounderLogistic]:
    """Fit fold-local cardiac/nuisance encoders and the custom guarded objective.

    Cardiac and nuisance PCA transforms are learned only from training patients.
    The custom model then optimizes disease discrimination while penalizing score
    covariance with the compressed nuisance axes.
    """
    x_encoder = CompactPCA(config.pca_variance, config.max_pca_components).fit(X)
    z_encoder = NuisanceEncoder(config.max_nuisance_components).fit(Z)
    Xc = x_encoder.transform(X)
    Zc = z_encoder.transform(Z)
    model = WorstCaseLinearConfounderLogistic(
        confounder_lambda=confounder_lambda,
        l2=l2,
        learning_rate=config.guard_learning_rate,
        max_iterations=config.guard_max_iterations,
        tolerance=config.guard_tolerance,
    ).fit(Xc, Zc, y)
    return x_encoder, z_encoder, model


def predict_guarded_logistic(
    fitted: tuple[CompactPCA, NuisanceEncoder, WorstCaseLinearConfounderLogistic],
    X: np.ndarray,
    Z: np.ndarray,
) -> np.ndarray:
    """Predict from cardiac features using a fitted confounder-guarded model.

    Nuisance variables are transformed only to verify the fold-local data
    contract; the deployed score itself depends on the cardiac representation
    and learned weights, not on acquisition metadata supplied at prediction.
    """
    x_encoder, z_encoder, model = fitted
    # Nuisance variables regularize training; prediction itself uses only the
    # cardiac representation, so no acquisition metadata is required at deploy time.
    _ = z_encoder.transform(Z)  # Validate the fold-local nuisance contract.
    return model.predict_proba(x_encoder.transform(X))[:, 1]


def fit_reliability_classifier(
    patient_bank: PatientBank, series_bank: SeriesBank, train_indices: np.ndarray,
    C: float, tau: float, config: Config,
) -> tuple[CompactPCA, LogisticRegression]:
    """Fit one row per training patient using the selected reliability weights.

    Because PCA/scaling/classifier are affine, averaging raw series logits with
    the same normalized weights is equivalent to classifying the weighted patient
    vector. Training directly on this vector avoids the old mismatch (equal-weight
    training followed by slice-count-weighted inference). No prior is estimated.
    """
    X = reliability_patient_embeddings(series_bank, patient_bank.patient_ids[train_indices], tau)
    return fit_standard_logistic(X, patient_bank.labels[train_indices], C, config)


def predict_reliability_patients(
    fitted: tuple[CompactPCA, LogisticRegression], series: SeriesBank,
    patient_ids: Sequence[str], tau: float,
) -> np.ndarray:
    """Classify reliability-pooled embeddings; equivalent to weighted raw logits."""
    return predict_standard_logistic(fitted, reliability_patient_embeddings(series, patient_ids, tau))


# =============================================================================
# 13. NESTED PATIENT-LEVEL CROSS-VALIDATION
# =============================================================================


def choose_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Choose a conservative Youden threshold from inner out-of-fold scores.

    The utility is sensitivity minus false-positive rate.  If several finite
    thresholds tie, the highest threshold is selected.  Outer-validation scores
    never influence the operating point.
    """
    false_positive_rate, true_positive_rate, thresholds = roc_curve(y_true, scores)
    finite = np.isfinite(thresholds)
    utility = true_positive_rate - false_positive_rate
    candidates = np.flatnonzero(finite & (utility == np.max(utility[finite])))
    # Among ties choose the highest threshold, which is more conservative.
    return float(np.max(thresholds[candidates]))


def candidate_grid(experiment: str, config: Config) -> list[dict[str, float]]:
    """Return a small predeclared grid; no expansion after inspecting outer results.

    The matched custom logistic uses the same code and L2 grid as the guarded
    method, but fixes lambda=0. Guard candidates must be strictly positive.
    rho=0 occurs only once per C (alpha is irrelevant when nothing is removed).
    """
    if experiment in {"BASELINE_A17", "CONTROL_SUPPORT_ONLY", "CONTROL_SHUFFLED_INTENSITY", "CONTROL_OUTSIDE_SUPPORT"}:
        return [{"C": C} for C in config.c_grid]
    if experiment == "CUSTOM_NUISANCE_PROJECTION":
        return [{"alpha": alpha, "rho": rho, "C": C}
                for rho in config.residual_rho_grid
                for alpha in ((max(config.residual_alpha_grid),) if rho == 0 else config.residual_alpha_grid)
                for C in config.c_grid]
    if experiment in {"CUSTOM_LOGISTIC_NO_GUARD", "CUSTOM_MINIMAX_CONFOUNDER_GUARD"}:
        lambdas = (0.0,) if experiment == "CUSTOM_LOGISTIC_NO_GUARD" else config.guard_lambda_grid
        return [{"lambda": lam, "l2": l2} for lam in lambdas for l2 in config.guard_l2_grid]
    if experiment == "CUSTOM_RELIABILITY_WEIGHTED_LOGIT_FUSION":
        return [{"C": C, "tau": tau} for C in config.c_grid for tau in config.fusion_tau_grid]
    raise KeyError(experiment)


def method_feature_mode(experiment: str) -> str:
    """Map a known experiment to its only frozen classifier feature mode."""
    if experiment not in EXPERIMENTS:
        raise KeyError(experiment)
    return {"CONTROL_SUPPORT_ONLY": "support_only",
            "CONTROL_SHUFFLED_INTENSITY": "shuffled_intensity",
            "CONTROL_OUTSIDE_SUPPORT": "outside_support"}.get(experiment, "cardiac")


def fit_predict_method(
    experiment: str, params: dict[str, float], patient_bank: PatientBank,
    series_bank: SeriesBank, train_indices: np.ndarray, valid_indices: np.ndarray,
    config: Config, diagnostics: dict[str, Any] | None = None,
) -> np.ndarray:
    """Fit ONLY train_indices; score ONLY valid_indices, enforcing disjoint patients.

    Fixed pretrained encoders and per-patient pooling precede CV, but all learned
    scalers, PCA, nuisance regression and classifiers are fitted here. The custom
    no-guard and guarded variants share every solver setting except lambda.
    """
    train_indices = np.asarray(train_indices, dtype=int)
    valid_indices = np.asarray(valid_indices, dtype=int)
    if set(train_indices.tolist()) & set(valid_indices.tolist()):
        raise ValueError("Training and validation patient rows overlap.")
    y_train = patient_bank.labels[train_indices]
    if experiment == "CUSTOM_RELIABILITY_WEIGHTED_LOGIT_FUSION":
        fitted = fit_reliability_classifier(patient_bank, series_bank, train_indices,
                                            params["C"], params["tau"], config)
        return predict_reliability_patients(fitted, series_bank, patient_bank.patient_ids[valid_indices], params["tau"])
    mode = method_feature_mode(experiment)
    X_train, X_valid = patient_bank.features[mode][train_indices], patient_bank.features[mode][valid_indices]
    if experiment in {"BASELINE_A17", "CONTROL_SUPPORT_ONLY", "CONTROL_SHUFFLED_INTENSITY", "CONTROL_OUTSIDE_SUPPORT"}:
        return predict_standard_logistic(fit_standard_logistic(X_train, y_train, params["C"], config), X_valid)
    Z = nuisance_block(patient_bank, config.regularization_nuisance_scope)
    Z_train, Z_valid = Z[train_indices], Z[valid_indices]
    if experiment == "CUSTOM_NUISANCE_PROJECTION":
        fitted = fit_residualized_logistic(X_train, Z_train, y_train, params["alpha"], params["C"], config, rho=params["rho"])
        return predict_residualized_logistic(fitted, X_valid, Z_valid)
    if experiment in {"CUSTOM_LOGISTIC_NO_GUARD", "CUSTOM_MINIMAX_CONFOUNDER_GUARD"}:
        fitted = fit_guarded_logistic(X_train, Z_train, y_train, params["lambda"], params["l2"], config)
        model = fitted[2]
        if diagnostics is not None:
            diagnostics.update(lambda_active=bool(params["lambda"] > 0),
                               optimizer_accepted_steps=len(model.training_trace) - 1,
                               initial_loss=float(model.training_trace[0]), final_loss=float(model.training_trace[-1]),
                               optimizer_max_iterations=config.guard_max_iterations,
                               reached_iteration_cap=len(model.training_trace) - 1 >= config.guard_max_iterations)
        return predict_guarded_logistic(fitted, X_valid, Z_valid)
    raise KeyError(experiment)


def parameter_preference(experiment: str, params: dict[str, float]) -> tuple:
    """Prefer less removal/count dependence when validation evidence is tied.

    alpha is ridge shrinkage, not removal fraction: rho explicitly controls removal.
    tau grows relative long-series preference, so SMALL tau wins a fusion tie.
    lambda=0 is available only in the separate matched no-guard experiment.
    """
    if experiment == "CUSTOM_NUISANCE_PROJECTION":
        return (params["rho"], -params["alpha"], params["C"])
    if experiment in {"CUSTOM_LOGISTIC_NO_GUARD", "CUSTOM_MINIMAX_CONFOUNDER_GUARD"}:
        return (params["lambda"], -params["l2"])
    if experiment == "CUSTOM_RELIABILITY_WEIGHTED_LOGIT_FUSION":
        return (params["tau"], params["C"])
    return (params["C"],)


def stratified_partitions(
    labels: np.ndarray,
    requested_folds: int,
    seed: int,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Construct patient folds, reducing their number only in tiny nested subsets.

    Reducing an inner fold count to the minority-class count is documented and
    label-stratified, not score-driven. Fewer than two patients per class cannot
    support a crossfit and causes a clear failure rather than fabricated scores.
    """
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=2)
    n_splits = min(requested_folds, int(counts.min()))
    if n_splits < 2:
        raise ValueError(f"Need at least two patients per class; counts={counts.tolist()}")
    return list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(np.zeros(len(labels)), labels))


def fixed_parameter_oof(
    experiment: str, params: dict[str, float], patient_bank: PatientBank,
    series_bank: SeriesBank, indices: np.ndarray, config: Config, seed: int,
) -> np.ndarray:
    """Generate score targets using ONLY a supplied training subset.

    Hyperparameters are held fixed; every target row is scored by a model fitted
    on other rows of this subset. This extra crossfit provides training targets
    for a nuisance-to-score emulator WITHOUT using any surrounding validation
    patient in the score generator. It never retunes on the surrounding test set.
    """
    values = np.full(len(indices), np.nan, dtype=np.float64)
    for train, valid in stratified_partitions(patient_bank.labels[indices], config.inner_folds, seed):
        values[valid] = fit_predict_method(experiment, params, patient_bank, series_bank,
                                          indices[train], indices[valid], config)
    if not np.isfinite(values).all():
        raise RuntimeError("Incomplete training-local OOF score targets.")
    return values


def fit_nuisance_score_probe(
    Z_train: np.ndarray, training_oof_scores: np.ndarray, Z_valid: np.ndarray,
    component_cap: int, ridge_alpha: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Fit a score emulator on training-only targets; predict held-out scores.

    Scaling, nuisance PCA, ridge coefficients and the constant reference are
    learned exclusively from Z_train and training_oof_scores. No held-out score
    or label is accepted by this function. Both outputs are forecasts of model
    scores, NOT estimates of disease probability from nuisance variables.
    """
    if not np.isfinite(training_oof_scores).all():
        raise ValueError("Invalid nuisance training targets.")
    encoder = NuisanceEncoder(component_cap).fit(Z_train)
    model = Ridge(alpha=ridge_alpha).fit(encoder.transform(Z_train), training_oof_scores)
    prediction = model.predict(encoder.transform(Z_valid))
    reference = np.full(len(Z_valid), float(np.mean(training_oof_scores)), dtype=np.float64)
    return np.asarray(prediction, dtype=np.float64), reference


def r2_diagnostics(
    target: np.ndarray,
    predicted: np.ndarray,
    reference: np.ndarray,
) -> dict[str, float | None]:
    """Report conventional R2 AND prediction skill versus a training-only mean.

    R2 = 1 - SSE/SST, where SST uses the pooled observed score mean. Prediction
    skill = 1 - SSE/SSE_reference, where each reference was predicted from that
    fold's training OOF score mean. A constant target/reference is reported as
    undefined (None), not forced to zero/one. Negative R2 means a weak surrogate,
    not negative confounding or demonstrated independence.
    """
    target, predicted, reference = [np.asarray(x, dtype=np.float64) for x in (target, predicted, reference)]
    if target.shape != predicted.shape or target.shape != reference.shape:
        raise ValueError("Nuisance diagnostic arrays must align.")
    if not all(np.isfinite(x).all() for x in (target, predicted, reference)):
        raise ValueError("Nuisance diagnostic arrays must be finite.")
    sse = float(np.sum((target - predicted) ** 2))
    sst = float(np.sum((target - target.mean()) ** 2))
    reference_sse = float(np.sum((target - reference) ** 2))
    return {"r2": None if sst <= 1e-12 else 1.0 - sse / sst,
            "predictive_r2": None if reference_sse <= 1e-12 else 1.0 - sse / reference_sse,
            "mse": sse / max(1, len(target)), "reference_mse": reference_sse / max(1, len(target))}


def inner_candidate_nuisance_skill(
    experiment: str, params: dict[str, float], scores: np.ndarray,
    outer_train_indices: np.ndarray, partitions: list[tuple[np.ndarray, np.ndarray]],
    patient_bank: PatientBank, series_bank: SeriesBank, config: Config, seed: int,
) -> dict[str, float | None]:
    """Evaluate nuisance predictability for one eligible candidate inside training.

    For each inner split, score targets for its TRAIN portion come from an extra
    crossfit confined to that portion. Fit Z->target there and predict the inner
    validation scores. The surrounding outer-test cohort is never read. This
    extra nesting avoids using scores generated by models that trained on the
    probe's validation patients, the flaw in the previous global-OOF R2 audit.
    """
    Z = nuisance_block(patient_bank, config.regularization_nuisance_scope)
    predicted, reference = np.full(len(scores), np.nan), np.full(len(scores), np.nan)
    for fold, (train, valid) in enumerate(partitions):
        inner_train, inner_valid = outer_train_indices[train], outer_train_indices[valid]
        targets = fixed_parameter_oof(experiment, params, patient_bank, series_bank,
                                      inner_train, config, seed + 2000 + fold)
        predicted[valid], reference[valid] = fit_nuisance_score_probe(
            Z[inner_train], targets, Z[inner_valid], max(config.audit_pca_components), config.audit_ridge_alpha)
    return r2_diagnostics(scores, predicted, reference)


def select_hyperparameters(
    experiment: str, patient_bank: PatientBank, series_bank: SeriesBank,
    outer_train_indices: np.ndarray, config: Config, seed: int,
) -> tuple[dict[str, float], np.ndarray, np.ndarray, list[dict[str, Any]]]:
    """Select on inner AUROC, then training-local nuisance skill when configured.

    First evaluate every candidate on the same inner folds. Restrict to those
    within auc_tolerance of best AUROC. For partial residualization and guarded
    logistic (with AND without guard), evaluate training-local nuisance prediction skill for eligible
    candidates. Prefer its lowest nonnegative value, within a declared tolerance;
    then use conservative parameter_preference. Negative skill gets no extra
    reward. Baseline/controls/fusion use AUROC plus deterministic ties.

    The returned OOF targets and selected parameters belong entirely to outer
    training. They can train the final outer-fold nuisance probe without seeing
    any outer-validation row. Model-score R2 is exploratory, not a causal metric.
    """
    outer_train_indices = np.asarray(outer_train_indices, dtype=int)
    labels = patient_bank.labels[outer_train_indices]
    partitions = stratified_partitions(labels, config.inner_folds, seed)
    candidates = []
    for params in candidate_grid(experiment, config):
        scores = np.full(len(labels), np.nan)
        for train, valid in partitions:
            scores[valid] = fit_predict_method(experiment, params, patient_bank, series_bank,
                                              outer_train_indices[train], outer_train_indices[valid], config)
        if not np.isfinite(scores).all():
            raise RuntimeError(f"Invalid inner scores: {experiment} {params}")
        candidates.append({"params": params, "inner_auc": float(roc_auc_score(labels, scores)),
                           "scores": scores, "nuisance_prediction_skill": None,
                           "nuisance_r2": None, "nuisance_selection_evaluated": False})
    best_auc = max(row["inner_auc"] for row in candidates)
    eligible = [row for row in candidates if row["inner_auc"] >= best_auc - config.auc_tolerance]
    if config.use_two_objective_selection and experiment in {
        "CUSTOM_NUISANCE_PROJECTION", "CUSTOM_LOGISTIC_NO_GUARD", "CUSTOM_MINIMAX_CONFOUNDER_GUARD"
    }:
        for row in eligible:
            diagnostic = inner_candidate_nuisance_skill(experiment, row["params"], row["scores"],
                outer_train_indices, partitions, patient_bank, series_bank, config, seed)
            row["nuisance_prediction_skill"] = diagnostic["predictive_r2"]
            row["nuisance_r2"] = diagnostic["r2"]
            row["nuisance_selection_evaluated"] = True
        evaluable = [row for row in eligible if row["nuisance_prediction_skill"] is not None]
        if evaluable:
            best_penalty = min(max(0.0, row["nuisance_prediction_skill"]) for row in evaluable)
            eligible = [row for row in evaluable if max(0.0, row["nuisance_prediction_skill"])
                        <= best_penalty + config.nuisance_skill_tolerance]
    selected = min(eligible, key=lambda row: parameter_preference(experiment, row["params"]))
    audit = [{k: v for k, v in row.items() if k != "scores"} | {
        "within_auc_tolerance": row["inner_auc"] >= best_auc - config.auc_tolerance,
        "selected": row is selected,
        "nuisance_scope": config.regularization_nuisance_scope,
    } for row in candidates]
    return selected["params"], labels, selected["scores"], audit


def nested_cv_once(
    experiment: str, patient_bank: PatientBank, series_bank: SeriesBank,
    config: Config, seed: int,
) -> dict[str, Any]:
    """Generate patient OOF disease scores AND strictly training-local nuisance forecasts.

    Each outer fold chooses parameters on inner CV, trains the disease predictor,
    and scores its held-out patients once. Independently fit each nuisance probe
    on the chosen INNER OOF training scores, never on OOF scores from other OUTER
    folds. Such other-fold generators may have trained on current test patients.

    Every cap/scope is evaluated prospectively. The largest configured cap is the
    primary nuisance diagnostic, not whichever looks most favorable. Test scores
    enter ONLY metric calculation after both disease and nuisance forecasts exist.
    The surrogate learns scores of smaller inner models and tests scores of a
    larger outer model; this distribution mismatch remains a limitation.
    """
    labels = patient_bank.labels
    if int(np.bincount(labels, minlength=2).min()) < config.outer_folds:
        raise ValueError("Not enough patients per class for the requested outer folds.")
    scores = np.full(len(labels), np.nan)
    predictions = np.full(len(labels), -1, dtype=np.int64)
    thresholds = np.full(len(labels), np.nan)
    folds = np.full(len(labels), -1, dtype=np.int64)
    probes = {f"{scope}_pc{cap}": {"scope": scope, "cap": cap,
              "predicted": np.full(len(labels), np.nan), "reference": np.full(len(labels), np.nan)}
              for scope in ("export", "anatomy_proxy", "combined") for cap in config.audit_pca_components}
    selected_params = []
    for fold, (train, valid) in enumerate(stratified_partitions(labels, config.outer_folds, seed), start=1):
        params, inner_labels, inner_scores, candidate_audit = select_hyperparameters(
            experiment, patient_bank, series_bank, train, config, seed + 1000 + fold)
        threshold = choose_threshold(inner_labels, inner_scores)
        optimization = {}
        fold_scores = fit_predict_method(experiment, params, patient_bank, series_bank, train, valid, config,
                                         diagnostics=optimization)
        scores[valid] = fold_scores
        thresholds[valid] = threshold
        predictions[valid] = (fold_scores >= threshold).astype(np.int64)
        folds[valid] = fold
        for probe in probes.values():
            Z = nuisance_block(patient_bank, probe["scope"])
            probe["predicted"][valid], probe["reference"][valid] = fit_nuisance_score_probe(
                Z[train], inner_scores, Z[valid], probe["cap"], config.audit_ridge_alpha)
        selected_params.append({"outer_fold": fold, "params": params,
            "inner_auc": float(roc_auc_score(inner_labels, inner_scores)), "threshold": threshold,
            "train_patient_ids": patient_bank.patient_ids[train].tolist(),
            "validation_patient_ids": patient_bank.patient_ids[valid].tolist(),
            "nuisance_training_targets": "selected inner OOF scores from outer-training patients only",
            "candidate_search": candidate_audit, "optimizer": optimization})
    if not np.isfinite(scores).all() or np.any(predictions < 0):
        raise RuntimeError(f"Incomplete OOF predictions for {experiment}")
    metrics = compute_metrics(labels, scores, predictions)
    for name, probe in probes.items():
        probe["metrics"] = r2_diagnostics(scores, probe["predicted"], probe["reference"])
        metrics[f"nuisance_{name}_r2"] = probe["metrics"]["r2"]
        metrics[f"nuisance_{name}_predictive_r2"] = probe["metrics"]["predictive_r2"]
    cap = max(config.audit_pca_components)
    for scope in ("export", "anatomy_proxy", "combined"):
        metrics[f"nuisance_{scope}_r2"] = metrics[f"nuisance_{scope}_pc{cap}_r2"]
        metrics[f"nuisance_{scope}_predictive_r2"] = metrics[f"nuisance_{scope}_pc{cap}_predictive_r2"]
    # Keep the familiar name, but document that the audit procedure has changed.
    metrics["nuisance_score_r2"] = metrics["nuisance_combined_r2"]
    return {"experiment": experiment, "seed": int(seed), "patient_ids": patient_bank.patient_ids.copy(),
            "labels": labels.copy(), "scores": scores, "predictions": predictions,
            "thresholds": thresholds, "folds": folds, "selected_params": selected_params,
            "metrics": metrics, "nuisance_probes": probes}


def compute_metrics(
    labels: np.ndarray,
    scores: np.ndarray,
    predictions: np.ndarray,
) -> dict[str, float]:
    """Compute patient-level discrimination and threshold metrics.

    AUROC and AUPRC use continuous out-of-fold scores.  Sensitivity, specificity,
    and F1 use the fold-specific thresholds chosen from training data.  Metrics
    describe the released cohort and are not clinical operating estimates.
    """
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    sensitivity = tp / max(1, tp + fn)
    specificity = tn / max(1, tn + fp)
    precision = tp / max(1, tp + fp)
    f1 = 2 * precision * sensitivity / max(1e-12, precision + sensitivity)
    return {
        "auc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1),
    }


def bootstrap_auc_interval(
    labels: np.ndarray,
    scores: np.ndarray,
    replicates: int,
    seed: int,
) -> tuple[float, float]:
    """Estimate a percentile AUROC interval by resampling patients.

    Each bootstrap replicate samples patient rows with replacement.  Replicates
    containing only one class are skipped because AUROC is undefined.  The
    interval quantifies finite-cohort sampling uncertainty but does not account
    for external domain shift.
    """
    rng = np.random.default_rng(seed)
    values = []
    indices = np.arange(len(labels))
    for _ in range(replicates):
        sample = rng.choice(indices, size=len(indices), replace=True)
        if len(np.unique(labels[sample])) < 2:
            continue
        values.append(roc_auc_score(labels[sample], scores[sample]))
    if not values:
        return float("nan"), float("nan")
    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))


# =============================================================================
# 14. RUN, SUMMARIZE, AND SAVE
# =============================================================================


def save_oof_result(result: dict[str, Any], output_dir: Path) -> None:
    """Save disease scores, training-only nuisance forecasts and all selected parameters.

    Called for the main run AND every repeated seed. This makes paired deltas and
    lambda/rho/tau frequencies reproducible from saved artifacts, not just console
    medians. Nuisance reference columns are outer-training score means; they are
    never estimated from the current held-out cohort.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    probes = result["nuisance_probes"]
    columns = ["patient_id", "label", "fold", "score", "threshold", "prediction"]
    columns += [f"{name}_{kind}" for name in probes for kind in ("predicted_score", "training_mean_reference")]
    with (output_dir / f"oof_{result['experiment']}.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        for i, pid in enumerate(result["patient_ids"]):
            values = [pid, int(result["labels"][i]), int(result["folds"][i]),
                      float(result["scores"][i]), float(result["thresholds"][i]), int(result["predictions"][i])]
            for probe in probes.values():
                values += [float(probe["predicted"][i]), float(probe["reference"][i])]
            writer.writerow(values)
    write_json(output_dir / f"selected_params_{result['experiment']}.json", result["selected_params"])
    write_json(output_dir / f"metrics_{result['experiment']}.json", result["metrics"])


def descriptive_summary(values: Sequence[float | None]) -> dict[str, Any]:
    """Summarize finite repeated values without treating repeats as independent cohorts."""
    x = np.asarray([float(v) for v in values if v is not None and np.isfinite(v)], dtype=np.float64)
    if not len(x):
        return {"runs": 0, "median": None, "q25": None, "q75": None, "minimum": None, "maximum": None}
    return {"runs": int(len(x)), "median": float(np.median(x)), "q25": float(np.quantile(x, 0.25)),
            "q75": float(np.quantile(x, 0.75)), "minimum": float(x.min()), "maximum": float(x.max())}


def paired_method_comparison(
    rows: list[dict[str, Any]],
    reference: str,
    comparison: str,
) -> dict[str, Any]:
    """Compare same-seed AUROC AND nuisance metrics; lower nuisance delta is preferable.

    Each statistic is computed on per-seed differences, never by subtracting two
    unrelated medians. Undefined surrogate R2 values remain missing and valid
    pair counts are reported. Fractions are descriptive, not binomial p-values.
    """
    left = {r["seed"]: r for r in rows if r["experiment"] == reference}
    right = {r["seed"]: r for r in rows if r["experiment"] == comparison}
    if set(left) != set(right):
        raise RuntimeError("Paired comparisons require the same repeat seeds.")
    metrics = ["auc", "nuisance_score_r2", "nuisance_export_r2", "nuisance_anatomy_proxy_r2",
               "nuisance_combined_predictive_r2", "nuisance_export_predictive_r2"]
    result = {"reference": reference, "comparison": comparison,
              "definition": "comparison minus reference on the same repeat seed", "metrics": {}}
    for metric in metrics:
        delta = [right[seed][metric] - left[seed][metric] for seed in sorted(left)
                 if right[seed].get(metric) is not None and left[seed].get(metric) is not None]
        summary = descriptive_summary(delta)
        summary["fraction_above_zero"] = float(np.mean(np.asarray(delta) > 0)) if delta else None
        summary["fraction_below_zero"] = float(np.mean(np.asarray(delta) < 0)) if delta else None
        result["metrics"][metric] = summary
    return result


def format_metric(value: float | None) -> str:
    """Format undefined score-dependence diagnostics explicitly in console output."""
    return "undefined" if value is None or not np.isfinite(value) else f"{value:.4f}"


def run_research_portfolio(config: Config) -> dict[str, Any]:
    """Run the small, auditable portfolio with isolated reports and a reusable cache.

    STAGES: validate -> preserve source/config -> audit series grouping -> frozen
    feature bank -> optional manually reviewed common cells -> separate nuisance
    blocks -> eight nested-CV experiments -> repeat all diagnostics -> save paired
    AUROC/R2 deltas. No model is promoted as an external/clinical success here.

    All reports use a fresh run subdirectory to avoid mixing old method names or
    metrics with this revised protocol. Expensive frozen features are cached in
    a sibling feature_cache directory. main() still reads no command-line args.
    """
    # STAGE 1: fail before downloads or image decoding on malformed configuration.
    validate_config(config)
    seed_everything(config.seed)
    output_root = Path(config.output_dir)
    tag = hashlib.sha256(json.dumps(asdict(config), sort_keys=True).encode()).hexdigest()[:8]
    run_name = f"run_{time.strftime('%Y%m%dT%H%M%S')}_{time.time_ns() % 1_000_000_000:09d}_{tag}"
    output_dir = output_root / run_name
    output_dir.mkdir(parents=True, exist_ok=False)
    config = replace(config, output_dir=str(output_dir))
    started = time.perf_counter()
    print(f"[RUN] revision={PIPELINE_REVISION}; results={output_dir}", flush=True)
    # STAGE 2: runtime metadata is notebook-safe even when __file__ is absent.
    write_json(output_dir / "configuration.json", asdict(config))
    write_json(output_dir / "run_metadata.json", {
        "revision": PIPELINE_REVISION, "cache_schema": FEATURE_CACHE_SCHEMA,
        "python": platform.python_version(), "numpy": np.__version__, "opencv": cv2.__version__,
        "torch": torch.__version__, "torchvision": torchvision.__version__, "scikit_learn": sklearn.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "runtime_environment": config.runtime_environment, "dataset_path": str(Path(config.dataset_path).resolve()),
        "dataset_path_source": config.dataset_path_source, "output_directory": str(output_dir.resolve()),
        "output_directory_source": config.output_dir_source, **_runtime_source_identity(),
    })
    # STAGE 3: freeze membership, and expose every changed series assignment.
    records = discover_records(config.dataset_path, config.series_grouping)
    grouping_audit = write_series_grouping_audit(records, config)
    # STAGE 4: cache depends on pixels/settings/group membership, not classifier grids.
    cache_config = replace(config, output_dir=str(output_root / "feature_cache"))
    series_bank = extract_series_bank(cache_config, records)
    duplicate_path = Path(cache_config.output_dir) / "duplicate_audit.json"
    if duplicate_path.is_file():
        write_json(output_dir / "duplicate_audit.json", json.loads(duplicate_path.read_text(encoding="utf-8")))
    # STAGE 5: manual protocol control is optional and must not be invented.
    series_bank, annotation_status = apply_reviewed_series_annotations(series_bank, config)
    patient_bank = build_patient_bank(series_bank, config.nuisance_random_projection_dim, config.nuisance_random_projection_seed)
    if len(patient_bank.patient_ids) != 30:
        print(f"[WARNING] This cohort has {len(patient_bank.patient_ids)} patients, not the historical 30.", flush=True)
    write_json(output_dir / "nuisance_design.json", {
        "regularization_scope": config.regularization_nuisance_scope,
        "blocks": {name: {"shape": list(values.shape)} for name, values in patient_bank.nuisance_blocks.items()},
        "export_columns": [METADATA_NAMES[i] for i in [0, 1, 2, 3, 4, 5]] +
                          ["n_series", "n_slices", "mean_series_length", "max_series_length"],
        "anatomy_proxy": "fixed projections of support/outside embeddings plus MONAI-valid rate and support area",
        "causal_warning": "No block is certified to contain only non-biological confounders.",
        "audit_cap_values": list(config.audit_pca_components), "primary_audit_cap": max(config.audit_pca_components),
        "audit_target_generation": "inner OOF scores inside each outer-training cohort; never global outer OOF targets",
    })
    # STAGE 6: evaluate main AND repeated runs, saving parameters/probes each time.
    main_results: dict[str, dict[str, Any]] = {}
    repeated_rows: list[dict[str, Any]] = []
    parameters_flat: list[dict[str, Any]] = []
    for experiment in EXPERIMENTS:
        print(f"\n[EVALUATION] {experiment}", flush=True)
        main_result = nested_cv_once(experiment, patient_bank, series_bank, config, config.seed)
        low, high = bootstrap_auc_interval(main_result["labels"], main_result["scores"],
                                           config.bootstrap_replicates, config.seed + 50_000)
        main_result["metrics"].update(auc_ci_low=low, auc_ci_high=high)
        main_results[experiment] = main_result
        save_oof_result(main_result, output_dir)
        for selected in main_result["selected_params"]:
            parameters_flat.append({"experiment": experiment, "run_type": "main", "repeat": 0,
                "seed": config.seed, "outer_fold": selected["outer_fold"], "inner_auc": selected["inner_auc"],
                "params_json": json.dumps(selected["params"], sort_keys=True),
                "optimizer_json": json.dumps(selected["optimizer"], sort_keys=True)})
        for repeat in range(config.repeated_cv_runs):
            seed = config.seed + 10_000 + repeat
            result = nested_cv_once(experiment, patient_bank, series_bank, config, seed)
            save_oof_result(result, output_dir / "repeated" / f"seed_{seed}")
            repeated_rows.append({"experiment": experiment, "repeat": repeat + 1, "seed": seed, **result["metrics"]})
            for selected in result["selected_params"]:
                parameters_flat.append({"experiment": experiment, "run_type": "repeated", "repeat": repeat + 1,
                    "seed": seed, "outer_fold": selected["outer_fold"], "inner_auc": selected["inner_auc"],
                    "params_json": json.dumps(selected["params"], sort_keys=True),
                    "optimizer_json": json.dumps(selected["optimizer"], sort_keys=True)})
            if repeat == 0 or (repeat + 1) % 5 == 0 or repeat + 1 == config.repeated_cv_runs:
                print(f"[REPEAT] {repeat + 1}/{config.repeated_cv_runs} AUROC={result['metrics']['auc']:.4f} "
                      f"export_R2={format_metric(result['metrics']['nuisance_export_r2'])} "
                      f"combined_R2={format_metric(result['metrics']['nuisance_score_r2'])}", flush=True)
        print(f"[RESULT MAIN] AUROC={main_result['metrics']['auc']:.4f}; "
              f"export_R2={format_metric(main_result['metrics']['nuisance_export_r2'])}; "
              f"combined_R2={format_metric(main_result['metrics']['nuisance_score_r2'])}", flush=True)
    # STAGE 7: save raw repeated rows and every selected lambda/rho/tau, not only medians.
    for filename, rows in (("repeated_cv_results.csv", repeated_rows), ("selected_params_all_runs.csv", parameters_flat)):
        with (output_dir / filename).open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    # STAGE 8: keep main and repeated statistics visibly separate in both outputs.
    summary_rows = []
    for experiment in EXPERIMENTS:
        repeated = [r for r in repeated_rows if r["experiment"] == experiment]
        row = {"experiment": experiment, **{f"main_{k}": v for k, v in main_results[experiment]["metrics"].items()}}
        for metric in ("auc", "auprc", "nuisance_score_r2", "nuisance_export_r2", "nuisance_anatomy_proxy_r2",
                       "nuisance_combined_predictive_r2", "nuisance_export_predictive_r2"):
            row.update({f"repeated_{metric}_{k}": v for k, v in descriptive_summary([r[metric] for r in repeated]).items()})
        summary_rows.append(row)
    summary_rows.sort(key=lambda r: (-r["repeated_auc_median"], r["experiment"]))
    with (output_dir / "experiment_summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0]))
        writer.writeheader()
        writer.writerows(summary_rows)
    # STAGE 9: paired differences for baseline contrasts and the key optimizer ablation.
    paired = {experiment: paired_method_comparison(repeated_rows, "BASELINE_A17", experiment)
              for experiment in EXPERIMENTS if experiment != "BASELINE_A17"}
    guard_pair = paired_method_comparison(repeated_rows, "CUSTOM_LOGISTIC_NO_GUARD", "CUSTOM_MINIMAX_CONFOUNDER_GUARD")
    frequency = {}
    for experiment in EXPERIMENTS:
        counts: dict[str, int] = defaultdict(int)
        for row in parameters_flat:
            if row["experiment"] == experiment and row["run_type"] == "repeated":
                counts[row["params_json"]] += 1
        frequency[experiment] = [{"params": json.loads(k), "outer_folds_selected": v} for k, v in sorted(counts.items())]
    write_json(output_dir / "selected_parameter_frequencies.json", frequency)
    write_json(output_dir / "paired_comparisons.json", {"versus_baseline": paired, "guard_vs_no_guard": guard_pair})
    # STAGE 10: record limitations and explicitly distinguish this protocol from v1.
    report = {
        "revision": PIPELINE_REVISION, "output_directory": str(output_dir),
        "patient_count": len(patient_bank.patient_ids), "series_count": len(series_bank.series_ids),
        "discovered_slice_count": len(records), "evaluated_slice_count": int(series_bank.n_slices.sum()),
        "series_grouping_audit": grouping_audit, "sequence_view_analysis": annotation_status,
        "results_ranked_by_repeated_auc_median": summary_rows,
        "paired_comparisons_vs_baseline": paired, "guard_vs_matched_no_guard": guard_pair,
        "paired_repeated_auc_deltas_vs_baseline": {k: v["metrics"]["auc"] for k, v in paired.items()},
        "selected_parameter_frequencies": frequency,
        "nuisance_audit_protocol": "outer-training-local score targets; 3/5-component probes reported, not selected on test",
        "nuisance_regularization_scope": config.regularization_nuisance_scope,
        "external_validation": {"status": "NOT_PERFORMED", "reason": "No independent compatible cohort supplied."},
        "runtime_seconds": time.perf_counter() - started,
        "interpretation_guardrails": [
            "This rerun reuses previously explored patients; it is not independent prospective validation.",
            "Grouping and evaluation changes prevent attributing old-versus-new AUC differences to one algorithm.",
            "Repeated seeds describe split sensitivity, not independent patient samples or a significance test.",
            "A negative R2 indicates failed score prediction by that surrogate, not removal of all confounding.",
            "Nuisance probes learn scores of inner models and evaluate a larger outer-trained model; mismatch can affect R2.",
            "Export metadata can correlate with biology; anatomy_proxy explicitly may contain real anatomy.",
            "The custom guard must be assessed against CUSTOM_LOGISTIC_NO_GUARD, not only sklearn's different solver.",
            "A reduced model-score R2 does not automatically reduce the separately trained outside-support classifier AUC.",
            "No model is declared clinically validated or causally deconfounded by this report.",
        ],
    }
    write_json(output_dir / "final_report.json", report)
    print("\nFINAL RANKING BY REPEATED-CV MEDIAN AUROC (ALL R2 BELOW ARE REPEATED MEDIANS)", flush=True)
    for rank, row in enumerate(summary_rows, 1):
        print(f"{rank:2d}. {row['experiment']:<43} AUROC={row['repeated_auc_median']:.4f} "
              f"IQR=[{row['repeated_auc_q25']:.4f}, {row['repeated_auc_q75']:.4f}] "
              f"export_R2={format_metric(row['repeated_nuisance_export_r2_median'])} "
              f"combined_R2={format_metric(row['repeated_nuisance_score_r2_median'])}", flush=True)
    print(f"[PAIRED GUARD - NO GUARD] median dAUROC={guard_pair['metrics']['auc']['median']:+.4f}; "
          f"median dExport_R2={format_metric(guard_pair['metrics']['nuisance_export_r2']['median'])}", flush=True)
    print(f"[REPORT] {output_dir / 'final_report.json'}", flush=True)
    return report


def run_showcase(config: Config) -> dict[str, Any]:
    """Backward-compatible generic alias for ``run_research_portfolio``.

    It exists only so earlier notebooks continue to run.  All scientific logic
    remains in one implementation.
    """

    return run_research_portfolio(config)


# =============================================================================
# 15. NOTEBOOK-SAFE ENTRY POINT
# =============================================================================


def build_main_config() -> tuple[Config, ResolvedRuntimePaths]:
    """Resolve paths and construct the immutable configuration used by ``main``.

    This function intentionally accepts no command-line namespace and never reads
    ``sys.argv``.  Consequently, internal Jupyter/Colab arguments such as
    ``-f <kernel.json>`` cannot alter the pipeline or trigger a parser failure.  Edit the ``MAIN_*`` constants above, or use the documented
    environment variables, when a non-default runtime setting is required.
    """

    paths = resolve_runtime_paths(
        MAIN_DATASET_PATH_OVERRIDE,
        MAIN_OUTPUT_DIR_OVERRIDE,
    )
    config = Config(
        dataset_path=str(paths.dataset_path),
        output_dir=str(paths.output_dir),
        dataset_path_source=paths.dataset_source,
        output_dir_source=paths.output_source,
        runtime_environment=paths.runtime_environment,
        batch_size=int(MAIN_BATCH_SIZE),
        num_workers=int(MAIN_NUM_WORKERS),
        repeated_cv_runs=int(MAIN_REPEATED_CV_RUNS),
        force_rebuild_cache=bool(MAIN_FORCE_REBUILD_CACHE),
        use_amp=bool(MAIN_USE_AMP),
        series_grouping=MAIN_SERIES_GROUPING,
        regularization_nuisance_scope=MAIN_REGULARIZATION_NUISANCE_SCOPE,
        audit_pca_components=tuple(MAIN_AUDIT_COMPONENT_CAPS),
        series_annotations_path=(MAIN_SERIES_ANNOTATION_PATH or os.environ.get("CAD_SERIES_ANNOTATION_CSV") or None),
    )
    return config, paths


def main() -> None:
    """Run the complete pipeline without consuming any command-line arguments.

    The function is safe when this file is:

    * launched with ``python cad_mri_university_showcase.py``;
    * executed with IPython ``%run``;
    * pasted or executed inside Colab/Jupyter;
    * imported and called as ``module.main()``.

    Notebook kernels may leave arbitrary values in ``sys.argv``.  They are
    deliberately ignored.  Runtime paths still support automatic Kaggle/local
    discovery, source-code overrides, and environment-variable overrides.
    """

    config, paths = build_main_config()
    print_resolved_runtime_paths(paths)

    if MAIN_CHECK_PATHS_ONLY:
        print(
            "[PATHS] Validation completed. No model was loaded and no output "
            "directory was created.",
            flush=True,
        )
        return

    run_research_portfolio(config)


if __name__ == "__main__":
    main()
